# v14 - v13 arm-N fold-0 leg alone (step D diagnostic)

Design: `docs/research/v13-strategy-revision-design.md` revision 3.2, section 4.5 (D-015). The
round-2 teacher student (v13 arm N: ConvNeXt-tiny, target `0.5 soft + 0.5 v11 OOF`, v11-B recipe)
from **fold 0 only**, without the public anchor. Pre-registered rule: train folds 1-4 only if the
public LB is >= 0.933; otherwise own-model work stops. The threshold is a resource-allocation
policy, not a calibrated forecast (design 4.5).

**Provenance.** Derived from our own `notebooks/v12-submit.ipynb` (Kaggle `lingxd/v12-submit`
version 1). The leg script is v12's (v06 slot selection, v07 preprocessing and v08 model code),
renamed to `v14_leg.py`. Changes:

- Weights: a list of (version, arm, fold) in the configuration; here one model,
  `v13_fold_0_N_best.pt`, from the private dataset `lingxd/rsna-knee-v13-weights`. The notebook
  checks each file's version, arm and fold before inference.
- OOF check (shard 0): only reference studies whose fold has a model are used (here the fold-0
  reference study); it is predicted by that fold's model and compared with the v13 arm-N fold-0
  OOF (recorded, tolerance 0.01; not blocking).
- Submission: mean of the models' per-finding ranks (one model here, so its own ranks).
- Unchanged: reference check of the preprocessing on three training studies, K_infer 16, 0.5 for
  a study the leg could not process. GPU T4 x2, internet off, same pinned Kaggle image as v12.

In [ ]:
# v14 configuration
import json
import os
import subprocess
import sys
import time
from pathlib import Path

VERSION = 'v14'
T0 = time.time()
K_INFER = 16
WEIGHT_SPECS = [('v13', 'N', 0)]   # (version, arm, fold) of each model
LEG_TIME_LIMIT_S = 8.0 * 3600
WORK = Path('/kaggle/working')
print(f'{VERSION}: leg only, K_infer {K_INFER}, models {WEIGHT_SPECS}', flush=True)

In [ ]:
%%writefile /kaggle/working/v14_leg.py
"""v14 leg: inference of the configured fold models on one shard of the test studies (v10/v12 leg code)."""
import json
import math
import sys
import time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, Dataset

LABELS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA',
          'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
FOCAL = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus']
N_SLOTS = 5
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)[:, None, None]
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)[:, None, None]


def log(msg):
    print(f'[{time.strftime("%H:%M:%S")}] {msg}', flush=True)
import base64
import hashlib
import io
import os
from concurrent.futures import ProcessPoolExecutor

# ---------------------------------------------------------------- v07 constants (verbatim)
SLOTS = ['SAG_FS', 'SAG_NFS', 'COR_FS', 'COR_OTHER', 'AX_FS']
SLOT_PLANE = {'SAG_FS': 'Sagittal', 'SAG_NFS': 'Sagittal', 'COR_FS': 'Coronal',
              'COR_OTHER': 'Coronal', 'AX_FS': 'Axial'}
MAX_SLICES = 32
CROP_MM = 140.0
OUT_PX = 320
WINDOW_PCT = (0.5, 99.5)
JPEG_QUALITY = 92
SIDE_DEADZONE_MM = 20.0

# ---------------------------------------------------------------- v07 shared functions (verbatim)
import cv2
import numpy as np
import pydicom

HEADER_TAGS = ['ImagePositionPatient', 'ImageOrientationPatient', 'InstanceNumber',
               'PixelSpacing', 'Rows', 'Columns', 'Laterality']

def _floats(value, n):
    try:
        out = [float(v) for v in value]
    except Exception:
        return None
    return out if len(out) == n and all(np.isfinite(out)) else None

def order_slices(records):
    # records: list of dicts with name, ipp, iop, inst. Returns (ordered names, method, info).
    info = {}
    with_geom = [r for r in records if r['ipp'] is not None and r['iop'] is not None]
    if records and len(with_geom) == len(records):
        iop = np.asarray(with_geom[0]['iop'], dtype=np.float64)
        normal = np.cross(iop[:3], iop[3:])
        norm = np.linalg.norm(normal)
        if norm > 1e-6:
            normal = normal / norm
            pos = np.asarray([np.dot(r['ipp'], normal) for r in with_geom])
            info['n_unique_positions'] = int(len(np.unique(np.round(pos, 3))))
            info['obliquity_deg'] = float(np.degrees(np.arccos(min(1.0, np.abs(normal).max()))))
            iop_all = np.asarray([r['iop'] for r in with_geom], dtype=np.float64)
            info['iop_consistent'] = bool(np.allclose(iop_all, iop_all[0], atol=1e-3))
            if info['n_unique_positions'] >= 0.9 * len(records):
                order = np.argsort(pos, kind='stable')
                gaps = np.diff(pos[order])
                info['slice_gap_mm'] = float(np.median(np.abs(gaps))) if len(gaps) else float('nan')
                return [with_geom[i]['name'] for i in order], 'geometry', info
    if records and all(r['inst'] is not None for r in records):
        ordered = sorted(records, key=lambda r: (r['inst'], r['name']))
        return [r['name'] for r in ordered], 'instance_number', info
    return sorted(r['name'] for r in records), 'filename', info

SLOT_SPEC = [
    ('SAG_FS', 'Sagittal', True, 8),
    ('SAG_NFS', 'Sagittal', False, 6),
    ('COR_FS', 'Coronal', True, 8),
    ('COR_OTHER', 'Coronal', None, 4),
    ('AX_FS', 'Axial', True, 6),
]

def _rank_key(c):
    return (0 if c['geometry_ok'] else 1, -int(c['n_readable'] or 0), c['SeriesInstanceUID'])

def select_slots(study_series):
    # study_series: list of dicts with SeriesInstanceUID, Anatomical_Plane, Fluid_Sensitive,
    # geometry_ok, n_readable. Returns {slot: (series uid or None, duplicate flag, fallback flag)}.
    usable = [s for s in study_series if (s.get('n_readable') or 0) > 0]
    chosen, used = {}, set()
    for slot, plane, fluid_pref, _k in SLOT_SPEC:
        same_plane = sorted((s for s in usable if s['Anatomical_Plane'] == plane), key=_rank_key)
        if not same_plane:
            chosen[slot] = (None, False, False)
            continue
        unused = [s for s in same_plane if s['SeriesInstanceUID'] not in used]
        if fluid_pref is None:
            preferred = sorted(unused, key=lambda s: (int(s['Fluid_Sensitive']),) + _rank_key(s))
        else:
            preferred = [s for s in unused if bool(s['Fluid_Sensitive']) == fluid_pref]
        if preferred:
            pick, duplicate, fallback = preferred[0], False, False
        elif unused:
            pick, duplicate, fallback = unused[0], False, True
        else:
            pick, duplicate, fallback = same_plane[0], True, True
        used.add(pick['SeriesInstanceUID'])
        chosen[slot] = (pick['SeriesInstanceUID'], duplicate, fallback)
    return chosen

# Canonical patient-space directions (DICOM LPS: +x left, +y posterior, +z superior).
# Per plane: (column direction, row direction, slice direction); 'medial' depends on side.
CANONICAL = {
    'Sagittal': ('posterior', 'inferior', 'medial'),
    'Coronal': ('medial', 'inferior', 'posterior'),
    'Axial': ('medial', 'posterior', 'inferior'),
}

def direction_vector(name, side):
    if name == 'medial':
        # The right knee's medial side faces the patient's left (+x), and vice versa.
        return np.array([1.0, 0.0, 0.0]) if side == 'R' else np.array([-1.0, 0.0, 0.0])
    return {'posterior': np.array([0.0, 1.0, 0.0]),
            'inferior': np.array([0.0, 0.0, -1.0])}[name]

def series_headers(series_dir):
    # Read headers of every file; return records plus first-file metadata.
    names = sorted(p.name for p in series_dir.iterdir() if p.suffix.lower() == '.dcm')
    records, meta = [], None
    for name in names:
        ds = pydicom.dcmread(str(series_dir / name), stop_before_pixels=True, force=True,
                             specific_tags=HEADER_TAGS)
        try:
            inst = int(float(str(getattr(ds, 'InstanceNumber', '')).strip()))
        except ValueError:
            inst = None
        rec = {'name': name,
               'ipp': _floats(getattr(ds, 'ImagePositionPatient', None) or [], 3),
               'iop': _floats(getattr(ds, 'ImageOrientationPatient', None) or [], 6),
               'inst': inst}
        records.append(rec)
        if meta is None:
            meta = {'spacing': _floats(getattr(ds, 'PixelSpacing', None) or [], 2),
                    'rows': getattr(ds, 'Rows', None), 'cols': getattr(ds, 'Columns', None),
                    'laterality': str(getattr(ds, 'Laterality', '') or '').strip().upper()}
    return records, meta

def image_centre_x(rec, meta):
    if rec['ipp'] is None or rec['iop'] is None or not meta['spacing'] or not meta['rows'] or not meta['cols']:
        return None
    ipp, iop = np.asarray(rec['ipp']), np.asarray(rec['iop'])
    centre = (ipp + iop[:3] * meta['spacing'][1] * float(meta['cols']) / 2
              + iop[3:] * meta['spacing'][0] * float(meta['rows']) / 2)
    return float(centre[0])

def decide_side(tags, centre_xs):
    # Returns (side, source). Tag first, then geometry with a dead zone.
    letters = [t[0] for t in tags if t and t[0] in ('L', 'R')]
    if letters:
        return max(set(letters), key=letters.count), 'tag'
    if centre_xs:
        m = float(np.median(centre_xs))
        if abs(m) >= SIDE_DEADZONE_MM:
            return ('R' if m < 0 else 'L'), 'geometry'
    return 'L', 'unresolved'

def geometry_side(centre_xs):
    if not centre_xs:
        return None
    m = float(np.median(centre_xs))
    return None if abs(m) < SIDE_DEADZONE_MM else ('R' if m < 0 else 'L')

def crop_resize(arr, spacing):
    # Centre crop of CROP_MM (zero padding if smaller), resize to OUT_PX.
    rows, cols = arr.shape
    want_r = int(round(CROP_MM / spacing[0])) if spacing else rows
    want_c = int(round(CROP_MM / spacing[1])) if spacing else cols
    out = np.zeros((want_r, want_c), dtype=np.float32)
    src_r0, src_c0 = max(0, (rows - want_r) // 2), max(0, (cols - want_c) // 2)
    dst_r0, dst_c0 = max(0, (want_r - rows) // 2), max(0, (want_c - cols) // 2)
    h, w = min(rows, want_r), min(cols, want_c)
    out[dst_r0:dst_r0 + h, dst_c0:dst_c0 + w] = arr[src_r0:src_r0 + h, src_c0:src_c0 + w]
    interp = cv2.INTER_AREA if min(want_r, want_c) >= OUT_PX else cv2.INTER_LINEAR
    return cv2.resize(out, (OUT_PX, OUT_PX), interpolation=interp), (h, w)

def encode_jpeg(img_u8):
    ok, buf = cv2.imencode('.jpg', img_u8, [int(cv2.IMWRITE_JPEG_QUALITY), JPEG_QUALITY])
    if not ok:
        raise RuntimeError('JPEG encode failed')
    return buf.tobytes()

def decode_jpeg(data):
    return cv2.imdecode(np.frombuffer(data, dtype=np.uint8), cv2.IMREAD_GRAYSCALE)

def prepare_study(study, slot_series, image_root):
    # slot_series: {slot: SeriesInstanceUID or None}. Returns (per-slot dict, study info).
    # Per slot: {'jpeg': [bytes], 'raw_index': [int], 'n_series': int, ...} or None.
    uids = sorted({u for u in slot_series.values() if u})
    plane_of = {u: SLOT_PLANE[s] for s, u in slot_series.items() if u}
    headers, tags, centre_xs = {}, [], []
    for uid in uids:
        records, meta = series_headers(image_root / study / uid)
        ordered, method, _info = order_slices(records)
        by_name = {r['name']: r for r in records}
        headers[uid] = (ordered, method, meta, by_name)
        tags.append(meta['laterality'])
        mid = by_name[ordered[len(ordered) // 2]]
        cx = image_centre_x(mid, meta)
        if cx is not None:
            centre_xs.append(cx)
    side, side_source = decide_side(tags, centre_xs)
    info = {'side': side, 'side_source': side_source,
            'tag_side': next((t[0] for t in tags if t and t[0] in ('L', 'R')), ''),
            'geometry_side': geometry_side(centre_xs) or ''}

    stored = {}
    for uid in uids:
        ordered, method, meta, by_name = headers[uid]
        n = len(ordered)
        keep = list(range(n)) if n <= MAX_SLICES else \
            sorted(set(np.linspace(0, n - 1, MAX_SLICES).round().astype(int).tolist()))
        spacing = meta['spacing']
        crops, fields, kept, n_bad = [], [], [], 0
        for i in keep:
            try:
                ds = pydicom.dcmread(str(image_root / study / uid / ordered[i]), force=True)
                arr = ds.pixel_array.astype(np.float32)
                if arr.ndim != 2:
                    raise ValueError(f'unexpected shape {arr.shape}')
            except Exception:
                n_bad += 1   # an unreadable slice is skipped, not fatal for the study
                continue
            kept.append(i)
            slope = float(getattr(ds, 'RescaleSlope', 1) or 1)
            intercept = float(getattr(ds, 'RescaleIntercept', 0) or 0)
            arr = arr * slope + intercept
            img, (h, w) = crop_resize(arr, spacing)
            crops.append(img)
            rows, cols = arr.shape
            r0, c0 = max(0, (rows - h) // 2), max(0, (cols - w) // 2)
            fields.append(arr[r0:r0 + h, c0:c0 + w].ravel())
        if not kept:
            stored[uid] = None
            continue
        lo, hi = np.percentile(np.concatenate(fields), WINDOW_PCT)
        # Orientation from the first stored slice's geometry.
        rec = by_name[ordered[kept[0]]]
        flip_lr = flip_ud = reverse = False
        if method == 'geometry' and rec['iop'] is not None:
            iop = np.asarray(rec['iop'])
            row_dir, col_dir = iop[:3], iop[3:]   # along a row (columns increase), down a column (rows increase)
            normal = np.cross(row_dir, col_dir)
            plane = plane_of[uid]
            want_cols, want_rows, want_slices = (direction_vector(n_, side) for n_ in CANONICAL[plane])
            flip_lr = float(np.dot(row_dir, want_cols)) < 0
            flip_ud = float(np.dot(col_dir, want_rows)) < 0
            reverse = float(np.dot(normal, want_slices)) < 0
        imgs = []
        for img in crops:
            u8 = np.clip((img - lo) / max(hi - lo, 1e-6) * 255.0, 0, 255).astype(np.uint8)
            if flip_lr:
                u8 = u8[:, ::-1]
            if flip_ud:
                u8 = u8[::-1, :]
            imgs.append(np.ascontiguousarray(u8))
        raw_index = list(kept)
        if reverse:
            imgs, raw_index = imgs[::-1], raw_index[::-1]
        stored[uid] = {'jpeg': [encode_jpeg(x) for x in imgs], 'raw_index': raw_index,
                       'n_series': n, 'sort_method': method, 'flip_lr': flip_lr,
                       'flip_ud': flip_ud, 'reverse': reverse, 'fov_ok': bool(spacing),
                       'n_decode_errors': n_bad,
                       'window': (float(lo), float(hi))}
    per_slot = {slot: (stored.get(uid) if uid else None) for slot, uid in slot_series.items()}
    return per_slot, info

# ----------------------------------------------------------------------------- data
def load_study(cache_dir, uid):
    with np.load(Path(cache_dir) / f'{uid}.npz') as z:
        return z['jpeg'], z['offsets'], z['slot']


def decode(jpeg, offsets, k):
    return cv2.imdecode(np.frombuffer(jpeg[offsets[k]:offsets[k + 1]].tobytes(), np.uint8),
                        cv2.IMREAD_GRAYSCALE)


def slot_centres(n, k, train, rng):
    if train:
        edges = np.linspace(0, n, k + 1)
        c = np.floor(edges[:-1] + rng.random(k) * (edges[1:] - edges[:-1])).astype(int)
    else:
        c = np.linspace(0, n - 1, k).round().astype(int)
    return np.clip(c, 0, n - 1)


def augment(tri, rng):
    # tri: (3, H, W) uint8 triplet; the same geometric transform for all three channels.
    h, w = tri.shape[1:]
    angle = rng.uniform(-10, 10)
    scale = rng.uniform(0.9, 1.1)
    tx, ty = rng.uniform(-0.05, 0.05) * w, rng.uniform(-0.05, 0.05) * h
    m = cv2.getRotationMatrix2D((w / 2, h / 2), angle, scale)
    m[:, 2] += (tx, ty)
    img = cv2.warpAffine(np.ascontiguousarray(tri.transpose(1, 2, 0)), m, (w, h),
                         flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    img = img.astype(np.float32) / 255.0
    gamma = rng.uniform(0.85, 1.15)
    img = np.clip(img, 0, 1) ** gamma
    img = img * rng.uniform(0.9, 1.1) + rng.uniform(-0.05, 0.05)
    return np.clip(img, 0, 1).transpose(2, 0, 1)


class StudyDataset(Dataset):
    def __init__(self, uids, targets, weights, cache_dir, k, train, img, seed):
        self.uids, self.targets, self.weights = list(uids), targets, weights
        self.cache_dir, self.k, self.train, self.img, self.seed = cache_dir, k, train, img, seed
        self.epoch = 0

    def __len__(self):
        return len(self.uids)

    def __getitem__(self, i):
        rng = np.random.default_rng((self.seed, self.epoch, i)) if self.train else None
        jpeg, offsets, slot = load_study(self.cache_dir, self.uids[i])
        tokens, slots, cache = [], [], {}
        for s in range(N_SLOTS):
            idx = np.flatnonzero(slot == s)
            if len(idx) == 0:
                continue
            n = len(idx)
            for c in slot_centres(n, self.k, self.train, rng):
                chans = []
                for d in (-1, 0, 1):
                    j = int(idx[min(max(c + d, 0), n - 1)])
                    if j not in cache:
                        cache[j] = decode(jpeg, offsets, j)
                    chans.append(cache[j])
                tri = np.stack(chans)
                if tri.shape[1] != self.img:
                    tri = np.stack([cv2.resize(x, (self.img, self.img), interpolation=cv2.INTER_AREA) for x in tri])
                x = augment(tri, rng) if self.train else tri.astype(np.float32) / 255.0
                tokens.append((x - MEAN) / STD)
                slots.append(s)
        return {'x': np.stack(tokens).astype(np.float32), 'slot': np.asarray(slots, np.int64),
                'y': self.targets[i], 'w': self.weights[i], 'uid': self.uids[i]}


def collate(batch):
    return batch


# ---------------------------------------------------------------------------- model
class Net(nn.Module):
    def __init__(self, backbone, pretrained, n_out=12, att_dim=256, drop=0.2):
        super().__init__()
        self.backbone = timm.create_model(backbone, pretrained=pretrained, num_classes=0)
        d = self.backbone.num_features
        self.slot_emb = nn.Embedding(N_SLOTS, d)
        self.norm = nn.LayerNorm(d)
        self.att_v = nn.Linear(d, att_dim)
        self.att_u = nn.Linear(d, att_dim)
        self.att_w = nn.Linear(att_dim, n_out)
        self.drop = nn.Dropout(drop)
        self.head_w = nn.Parameter(torch.zeros(n_out, d))
        self.head_b = nn.Parameter(torch.zeros(n_out))
        nn.init.normal_(self.head_w, std=0.01)

    def encode(self, x, chunk=None):
        if chunk is None or x.shape[0] <= chunk:
            return self.backbone(x)
        return torch.cat([self.backbone(x[i:i + chunk]) for i in range(0, x.shape[0], chunk)])

    def pool(self, h, slot, counts):
        # h: (N, D) tokens of several studies concatenated; counts: tokens per study.
        h = self.norm(h + self.slot_emb(slot))
        a = self.att_w(torch.tanh(self.att_v(h)) * torch.sigmoid(self.att_u(h))).float()  # (N, 12)
        out, start = [], 0
        for n in counts:
            hs, as_ = h[start:start + n].float(), a[start:start + n]
            wts = torch.softmax(as_, dim=0)                                  # (n, 12)
            pooled = torch.einsum('nk,nd->kd', wts, hs)                      # (12, D)
            out.append((self.drop(pooled) * self.head_w).sum(-1) + self.head_b)
            start += n
        return torch.stack(out)                                              # (B, 12)


def batch_tensors(batch, device):
    x = torch.from_numpy(np.concatenate([b['x'] for b in batch])).to(device, non_blocking=True)
    slot = torch.from_numpy(np.concatenate([b['slot'] for b in batch])).to(device)
    counts = [len(b['slot']) for b in batch]
    return x, slot, counts


# ---------------------------------------------------------------- leg (v09/v10 code)
ARGS = json.loads(sys.argv[1])
REFERENCE = json.loads(Path(ARGS['reference_json']).read_text())


def series_info(image_root, study, uid):
    # Header-only facts used by select_slots (v06 semantics: n_readable, geometry_ok).
    try:
        records, meta = series_headers(image_root / study / uid)
        _ordered, method, _info = order_slices(records)
        return {'n_readable': len(records), 'geometry_ok': method == 'geometry' and bool(meta['spacing'])}
    except Exception:
        return {'n_readable': 0, 'geometry_ok': False}


def pack(per_slot):
    # Same layout as v07 cache_one: slots in SLOTS order, a shared series stored per slot.
    blobs, offsets, slot_idx, raw_idx = [], [0], [], []
    for s, slot in enumerate(SLOTS):
        entry = per_slot[slot]
        if entry is None:
            continue
        for b, r in zip(entry['jpeg'], entry['raw_index']):
            blobs.append(b)
            offsets.append(offsets[-1] + len(b))
            slot_idx.append(s)
            raw_idx.append(r)
    return (np.frombuffer(b''.join(blobs), dtype=np.uint8), np.asarray(offsets, dtype=np.int64),
            np.asarray(slot_idx, dtype=np.int8), np.asarray(raw_idx, dtype=np.int16))


def prepare_one(job):
    study, series_rows, image_root, cache_dir = job
    try:
        rows = []
        for r in series_rows:
            rows.append({**r, **series_info(Path(image_root), study, r['SeriesInstanceUID'])})
        chosen = select_slots(rows)
        slot_series = {slot: chosen[slot][0] for slot in SLOTS}
        per_slot, info = prepare_study(study, slot_series, Path(image_root))
        jpeg, offsets, slot, raw = pack(per_slot)
        if len(slot) == 0:
            return study, 'no slices'
        np.savez(Path(cache_dir) / f'{study}.npz', jpeg=jpeg, offsets=offsets, slot=slot, raw_index=raw)
        return study, ''
    except Exception as exc:
        return study, f'{type(exc).__name__}: {str(exc)[:200]}'


def reference_check(comp, cache_dir):
    # Re-create three v07 training studies and compare with the stored fingerprints.
    train_series = pd.read_csv(comp / 'train_series.csv', dtype={'StudyInstanceUID': str, 'SeriesInstanceUID': str})
    if not (comp / 'train_series').is_dir():
        return {'status': 'skipped', 'reason': 'train_series not mounted'}
    out = {'status': 'pass', 'studies': {}}
    for uid, ref in REFERENCE.items():
        rows = train_series[train_series.StudyInstanceUID == uid][
            ['SeriesInstanceUID', 'Anatomical_Plane', 'Fluid_Sensitive']].to_dict('records')
        study, err = prepare_one((uid, rows, str(comp / 'train_series'), cache_dir))
        if err:
            out['studies'][uid] = {'error': err}
            out['status'] = 'fail'
            continue
        with np.load(Path(cache_dir) / f'{uid}.npz') as z:
            jpeg, offs, slot, raw = z['jpeg'], z['offsets'], z['slot'], z['raw_index']
        ref_arr = np.load(io.BytesIO(base64.b64decode(ref['arrays_b64'])))
        same_layout = np.array_equal(slot, ref_arr['slot']) and np.array_equal(raw, ref_arr['raw_index'])
        byte_equal = hashlib.sha256(jpeg.tobytes()).hexdigest() == ref['jpeg_sha256']
        max_diff = None
        if same_layout:
            thumbs = np.stack([cv2.resize(decode(jpeg, offs, k), (8, 8), interpolation=cv2.INTER_AREA)
                               for k in range(len(offs) - 1)]).astype(int)
            max_diff = int(np.abs(thumbs - ref_arr['thumbs'].astype(int)).max())
        ok = same_layout and (byte_equal or (max_diff is not None and max_diff <= ARGS['thumb_tolerance']))
        out['studies'][uid] = {'byte_equal': byte_equal, 'same_layout': same_layout, 'thumb_max_diff': max_diff}
        if not ok:
            out['status'] = 'fail'
        (Path(cache_dir) / f'{uid}.npz').unlink(missing_ok=True)
    return out


def oof_check(models, comp, cache_dir, device):
    # End-to-end check of the weights: re-create the reference training studies from DICOM,
    # predict each with the fold model that did not train on it, compare with that fold's OOF.
    expect = ARGS.get('oof_expect') or {}
    if not expect:
        return {'status': 'skipped', 'reason': 'no expectations'}
    if not (comp / 'train_series').is_dir():
        return {'status': 'skipped', 'reason': 'train_series not mounted'}
    train_series = pd.read_csv(comp / 'train_series.csv', dtype={'StudyInstanceUID': str, 'SeriesInstanceUID': str})
    ref_dir = Path(cache_dir) / 'oof_check'
    ref_dir.mkdir(parents=True, exist_ok=True)
    out = {'status': 'pass', 'tolerance': ARGS['oof_tolerance'], 'studies': {}}
    for uid, e in expect.items():
        rows = train_series[train_series.StudyInstanceUID == uid][
            ['SeriesInstanceUID', 'Anatomical_Plane', 'Fluid_Sensitive']].to_dict('records')
        _study, err = prepare_one((uid, rows, str(comp / 'train_series'), str(ref_dir)))
        if err:
            out['studies'][uid] = {'fold': e['fold'], 'error': err}
            out['status'] = 'fail'
            continue
        ds = StudyDataset([uid], np.zeros((1, 12), np.float32), np.ones((1, 12), np.float32), str(ref_dir),
                          ARGS['k_infer'], False, ARGS['img'], 0)
        x, slot, counts = batch_tensors([ds[0]], device)
        m = models[ARGS['weight_folds'].index(e['fold'])]
        with torch.no_grad(), torch.autocast(device_type=device.type, dtype=torch.float16,
                                             enabled=device.type == 'cuda'):
            logits = m.pool(m.encode(x, chunk=ARGS['eval_batch_tokens']), slot, counts)
        p = torch.sigmoid(logits.float()).cpu().numpy()[0]
        diff = float(np.abs(p - np.asarray(e['probs'], np.float32)).max())
        out['studies'][uid] = {'fold': e['fold'], 'max_abs_diff': round(diff, 6)}
        if not diff <= ARGS['oof_tolerance']:
            out['status'] = 'fail'
        (ref_dir / f'{uid}.npz').unlink(missing_ok=True)
    return out


def main():
    t0 = time.time()
    comp, out_dir = Path(ARGS['comp']), Path(ARGS['out_dir'])
    cache_dir = Path(ARGS['cache_dir'])
    cache_dir.mkdir(parents=True, exist_ok=True)
    out_dir.mkdir(parents=True, exist_ok=True)
    receipt = {'shard': ARGS['shard'], 'status': 'running', 'opencv': cv2.__version__, 'torch': torch.__version__}
    write = lambda: (out_dir / f'v14_leg_receipt_{ARGS["shard"]}.json').write_text(json.dumps(receipt, indent=2))
    if ARGS['shard'] == 0:
        receipt['reference_check'] = reference_check(comp, cache_dir)
        log(f'reference check: {json.dumps(receipt["reference_check"])}')
    test = pd.read_csv(comp / 'test.csv', dtype={'StudyInstanceUID': str})
    series = pd.read_csv(comp / 'test_series.csv', dtype={'StudyInstanceUID': str, 'SeriesInstanceUID': str})
    studies = sorted(test.StudyInstanceUID.unique())[ARGS['shard']::ARGS['n_shards']]
    by_study = {k: g[['SeriesInstanceUID', 'Anatomical_Plane', 'Fluid_Sensitive']].to_dict('records')
                for k, g in series.groupby('StudyInstanceUID')}
    jobs = [(s, by_study.get(s, []), str(comp / 'test_series'), str(cache_dir)) for s in studies]
    errors = {}
    with ProcessPoolExecutor(ARGS['workers']) as pool:
        for i, (study, err) in enumerate(pool.map(prepare_one, jobs, chunksize=2)):
            if err:
                errors[study] = err
            if (i + 1) % 100 == 0:
                log(f'shard {ARGS["shard"]}: prepared {i + 1}/{len(jobs)}')
            if time.time() - t0 > ARGS['time_limit_s']:
                break
    ready = [s for s in studies if s not in errors and (cache_dir / f'{s}.npz').is_file()]
    receipt.update(n_studies=len(studies), n_prepared=len(ready), prep_errors=dict(list(errors.items())[:20]),
                   prep_seconds=round(time.time() - t0, 1))
    log(f'shard {ARGS["shard"]}: prepared {len(ready)}/{len(studies)} in {receipt["prep_seconds"]}s')
    write()

    device = torch.device(ARGS.get('device', 'cuda'))
    models = []
    for f in ARGS['weights']:
        ck = torch.load(f, map_location='cpu', weights_only=False)
        m = Net(ck['cfg']['backbone'], False)
        m.load_state_dict(ck['model'])
        models.append(m.to(device).eval())
    if ARGS['shard'] == 0:
        receipt['oof_check'] = oof_check(models, comp, cache_dir, device)
        log(f'OOF check: {json.dumps(receipt["oof_check"])}')
        write()
    zeros, ones = np.zeros((len(ready), 12), np.float32), np.ones((len(ready), 12), np.float32)
    ds = StudyDataset(ready, zeros, ones, str(cache_dir), ARGS['k_infer'], False, ARGS['img'], 0)
    loader = DataLoader(ds, batch_size=1, shuffle=False, num_workers=ARGS['workers'], collate_fn=collate)
    rows, done = [], 0
    with torch.no_grad():
        for batch in loader:
            x, slot, counts = batch_tensors(batch, device)
            probs = []
            for m in models:
                with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == 'cuda'):
                    h = m.encode(x, chunk=ARGS['eval_batch_tokens'])
                    logits = m.pool(h, slot, counts)
                probs.append(torch.sigmoid(logits.float()).cpu().numpy()[0])
            rows.append([batch[0]['uid']] + np.concatenate(probs).tolist())
            done += 1
            if time.time() - t0 > ARGS['time_limit_s']:
                receipt['status'] = 'timeout'
                break
    cols = ['StudyInstanceUID'] + [f'{c}__f{k}' for k in range(len(models)) for c in LABELS]
    pd.DataFrame(rows, columns=cols).to_csv(out_dir / f'v14_leg_shard{ARGS["shard"]}.csv', index=False)
    if receipt['status'] == 'running':
        receipt['status'] = 'done'
    receipt.update(n_predicted=done, total_seconds=round(time.time() - t0, 1))
    write()
    log(json.dumps({k: v for k, v in receipt.items() if k != 'prep_errors'}))


if __name__ == '__main__':
    main()

In [ ]:
# Pixel decoders available in this image (for the record).
diag = {}
for mod in ('pylibjpeg', 'openjpeg', 'libjpeg', 'gdcm', 'PIL'):
    try:
        diag[mod] = getattr(__import__(mod), '__version__', 'present')
    except Exception:
        diag[mod] = None
try:
    from PIL import features
    diag['pil_jpg_2000'] = bool(features.check('jpg_2000'))
except Exception:
    diag['pil_jpg_2000'] = None
print(json.dumps(diag), flush=True)

def find_files(name, root=Path('/kaggle/input')):
    hits = []
    for current, dirs, files in os.walk(root):
        dirs[:] = [d for d in dirs if d not in ('train_series', 'test_series', 'competitions')
                   and 'rsna-knee-abnormality-detection' not in d]
        if name in files:
            hits.append(Path(current) / name)
    return hits

comp = next(p for p in (Path('/kaggle/input/competitions/rsna-knee-abnormality-detection'),
                        Path('/kaggle/input/rsna-knee-abnormality-detection'))
            if (p / 'test.csv').is_file())
import torch
weights, weights_meta = [], {}
for ver, arm, k in WEIGHT_SPECS:
    name = f'{ver}_fold_{k}_{arm}_best.pt'
    hits = find_files(name)
    assert len(hits) == 1, f'{name}: {hits}'
    cfg = torch.load(hits[0], map_location='cpu', weights_only=False)['cfg']
    assert (cfg['version'], cfg['arm'], cfg['fold']) == (ver, arm, k), f'{name}: {cfg["version"]}, {cfg["arm"]}, {cfg["fold"]}'
    weights.append(str(hits[0]))
    weights_meta[name] = {'epochs': cfg['epochs'], 'lr_backbone': cfg['lr_backbone'], 'img': cfg['img'], 'backbone': cfg['backbone']}
weight_folds = [k for _ver, _arm, k in WEIGHT_SPECS]
assert len(set(weight_folds)) == len(weight_folds), 'one model per fold'
print(json.dumps(weights_meta), flush=True)
OOF_EXPECT = json.loads('{"1.2.826.0.1.3680043.8.498.10018552945042470316597008531304016339": {"fold": 0, "probs": [0.074977, 0.039888, 0.074822, 0.047303, 0.071729, 0.063204, 0.216678, 0.18957, 0.240546, 0.116425, 0.074574, 0.126038]}}')   # v13 arm-N OOF of the reference studies in the folds with a model
assert all(e['fold'] in weight_folds for e in OOF_EXPECT.values())
ref = WORK / 'v14_reference.json'
ref.write_text('{"1.2.826.0.1.3680043.8.498.10004945927472656027199792075652399585": {"jpeg_sha256": "fb3ed5f17b44209ad7e667b6e9c61969fa7761bb806a9afe5318c54efe4fd6b4", "n": 156, "arrays_b64": "UEsDBC0AAAAIAAAAIQB9b6ZV//////////8KABQAdGh1bWJzLm5weQEAEACAJwAAAAAAADwiAAAAAAAAnZlXc2NJlt9v3gtDADQA6AkaECQMCZDwhPfee+8IECBIECBB0HuyyKIpX112urpr2s12j+mdmRi3LqQNaUO72pAi9AH0sA96VoS+gRLskbalCL0IwYhz3e9knrx58/xP8pU/5gumALKLHHHLla2lFlfN5B7viLkCJnd5o7XdKq4XNlrlSvu6vVjfqsDrWyvFZgWe88TyBQFTCf9mBMwT5v/PjwxGiv/w26+MKEL0nnz3mo8iqGTl/Q0XIOjg8YMDEUAAoXa6F0QRgM5fbajbFqdRzgCAYhiJQgaoKvzr737pxMCI5+U3n8swhBjef363ABBMX0+tedv+Lr22LWiJjUVVElp0yCVSYpDHdXdTULxx8cOT92E8Jks9fPZ+AQOc/PeNNxYUoWZalp0KihCazdjSFQ4Bs7tWwwq0qMJod+ABiuL76L14erz0+uZlltydLJ1dvPBhxPTL69LLEgbUu+e50hGGTJ1t5w+ekBBcolGpHuIA6LDYMwECimLdI2M0/IS3frt/WaRNWArvDi5zHSOJF/b4UR0PIulU0L/XAww71eD26TDoWd7dbl30AHTYGUyGewAO3z/GHiRJ3PW7/bOVIcnp8f7adoQmeJyxptdDvcRyJp+InjNAYqOS2j3mIszi8Vb1cBygfE80FhyF4dMmeMNkTTKhqByVxtTXUWEuKB2WbDks0Y3UxGArnogU3rLR2lEu37wQA9HyfrNyNgcweSQZdrFRHHFwik3rtCVMfF/BN6k+NCu8OvGYYtVsjKyFpmc3A2Ff4VMh8fAglt54pgLms2apeq5AiPpYPOKFfjoZHO4g3RM2enxBoVC3agmGTHKBuWZxB+pRqa7qCniKLw2j1VY4tvHGjzp2atmVGwfo9WWSJo8WxXdOTLNHGO640e+NO1WGJUcq4zLIbasWV7QWUZtrHp9v8bmds7gVjW28TmOWw3q+cu0CQ8FiSu4xAozK5oyMDHm9OqtjddliXnKk4367wV+32JMbCZNnzen0VZ4FRfn1bHL9RRVv3mksFs4igOEoJNVmG4qnc7ij9KFwTG3xNlec9oKlBN+YNbhmc8Wacbe74Q8EylcxeXlndXH9yTrBsbdRLRwkAMO+mDJZbQBHneUwaEOhmN6W3t8I2VPO9eZl3heueP3Zq0jIsRkPuc8vM8rKWauw8Wizw3W2tZrZhrxlMWExGQBGn5ud6O2NBnX6x2fNXCDhXjx72Yx7F0PBwieBqGUjEfE9382rlk6Oykvn2yTvg91151oaMIz5nFhlALheAX+0G/Ja9ZPLrXI45o0/enW86E1Fwo036YSlkUpufLpeUlaO3h49ud4n+04v9o62S2DYmFlKmrUoRpsRmsb7ox6V8vRga8Ufc/sieycr7mQkmLhZy5m3ctn1dw/XlI2TF5Xzq0Ny5PRqP7tZAqOWQsmsUMD2+TJ2FzXolguWDxsrnoxJ51jeKtsSLodzdzltrMdjjZ3DTUnj5DKRW93rCG+dNUKBHGAYYimDdAHg+2bmhmlUV0QlCJ5slD0xizK8vl22pV3O1INmQdeMOWr7h/uCrav9TL15RsisH+xECgUwZoznbHolwGhs7gC927TmFjvPtoqugFPub25XDBmbIXyxXZY3g87W7vkxZ3e/kVrfvsTnVnf24uUCGLUloi7zAorr4/GHqV2qVkKWPdsvmiNOWXhvv6RuOo3hs53qfCvsO7u5Oh3f38+nDndvscRa86BYWgZD1ljAblejaC+bM0ylCFtFq+7iqKAoB8SundO8KmnSBDcOa5MVr7NyeHvWv9OI1g9PnqCh5fL24lID9NrCSZdZAVefSc5oN5mzWV9Unp+EmPWC3Fg/TgqDVlVg9bDal7ebTbHLHcreWr6xef4CuMrp1ezKBuix+2JOgwKg/UzOeD957GBjR3N55uxuLZs1lVPfQNmmcWbPysSEz210XNSxlXKhnD9/AnSlVCGysoGQrcGYV68BaB+TPTnWNXDc2jPvHdmwlXpUlzvT44s2tdl3nUKDfofatLEEksVsMvrgCRCWkglPbR3B611Rj1oFeQZjcpCCPzret9QvJKj7Ys3puRKD8dSC3XisBsyCT2dPOoC4GnGFGweAtpdKuhurCKow29x6NQBdzNFJRjdaf7Cvdx7MosqrQ+f8vgCQyxq9taIDpIpH6wiqQP9W0BZeWgGE1lLOvZkDQOB1BnQKgJGYU+NDXaD49FgjMMyB2dsr7QyXD3BLVoVBpEHQjNfo8ugArhm1RpJJgKa2F02bSQSMRVxu7QIAuLHxsYFeEHh5Yk6J+GDw+qFUKp9BsFBErfEoENQftOXLcJzzGWtxLYgA223SWvMCQI3qIlo5zCP9Y+O9vUD/8iLSkvNA58GNoaDjAJylYYiEpQA1JYM7GQECzDuRypoJAVOP3PmCEebDQDSsXYD8AHMctj/34iZzop9G8OVbYys8haD8gqFWEcLnk6q1dTYAk4eRwqYcQTsvW6lVGeSMWY1aAm0ng9nbD8ZuHuWeuSYRXHDbdOpmIriRZWOjyEZQyra/XBtFAPG8WT3mIABbfNA65UGOF4kG5qAlj4wy6KDr6pPUS28fgmlLmiv3AEApe55mZQQm6Lu1lTUqfK7wydbBIIJiipvjEwa0A7F0gN3O42PMIRrw/GMwa9sCCO0PZ9WEB+b9VsXbUMK8L31j3lEjCEJ4uXGZhhbhr9VgXkcQlMnobxsMh2EI8vK9wS1rASTxq/WgBOb3zqdrnoU4goB1HV+xBJ/T7bMEV5DDJ1SMEPSLdHF7R6FFUSIOB0h/411wz7dQ7MuHzaA4gyKF6o1VkgHIaC2nmaohSMcDLUv8BvJs9xCvCPUHmGBOTrUtjkQkoOyn0wrv3DGJGUleLfJLgKzMPg1KkwhiDf50X3UEEH5T59H9AkNQq5ArXYYdxk0Mz8xCHiOSKETUVNjKxbirLBnzcmtjpowbXnj6actQA8B8+bNXmQ8ExBQp5NTPqEin1eKSLRERQBmdnp2BFuugUAi4eOh5bIMTNS0Ez4r76sbUROXzr85C74aIro9fvsksMxFruprXXPKRPrnfJwkPIaBncpY72d0Ov7OH1JGP7SRuOd6cdO/o0cHRtxZO9dOP14GXmp74q/fPM0E1cGUXM7K8ERmWBt1SEw9g/TN8DqMXYHhSZw+ZUMidvH2/2NpWHpx/ePDtt6X55PvPr51PUgOxu1e3MbUDF8+lkhJjEJmw+KxymQzBBufnuAMDCMCTuqgd5MLS1S++vvrVG33t+suH3//iuTb45sMjz4sH4+XDmwufsEQJpPweyXwSjHo9JjHbiODHpcLhPji/MEpnN7GzsPr0u292r7+0lK4/3L755jOL78WnD9JvX09V9u4uXcK17kDCZRfPRcC432cRcdQIcVIu7utnwHnd00PuoBbLz3/+XST4S/viow+3P/nsL5zeZ58drb3/fLa8eXdrkTRpobjdJBJE0Alf0DHHViIdkKf33vPdZGJ3qfTyq58ZNb/1Zp98+vLJxfde77MvNldffDlbWb87N9h2+uJRu0IiCGMsZ8A3z1bc8710yBOoPSQ8rVh+98UTEev7UOHZF9vO/O/84Sdf1UyvvptZqd1Fw/GzwVjcKbfMQ97miosm5QhpUirupbW/Szq1A9+zuPzmJwUB670/++zjqsr4c1/4yTcrcz/9S+7a6iO9OXw7HE1bNRZ+GMex25OiCRGMXyrt7YHfI7GHhsdTloufvF6ZnboJp59/bOjnX/sjz7/eSv7k++na2pXTk/iEEY1YjF5RGsd1OLKiaSkgsWSKQfogXD97aSRC12L4xettweSRP/nqqxcFwaUn9urbq/rVX3KWNy5jQeXT0YTHpAiK8zi+3Z2Ss8WANCleYPT3A5REo3d2dMQ0z189nWFt+WJvfvZ1kb3riL/59sxa+mZmcXNvSc0/Z6TsOmmIE8dmra6gdGgOIU/Mywa64XpDofUQcXjH/IOX3wmmWi7HJ998SE83LOFPvssqIh95me2dkIh3NJp0mhSWySAmtLj985MihDIhlA5TaQiORKURMLxhKnP5G/70tl35+Osnoem6yv7iZ1ax+z0nc3Di4fOOGXG7Wa5gBtB5ndM/xxIj5PF50Vh/L0A7qDQiRjBOa8t/mOc2bfL9L8597DpX++Irp8DxYiJy8jAp5J72BW3aWd2kD/CU9oByfA6QxgRzY4O9KEqm0TrweAWPa/sryUxdZst9OAryWkL1i68Wxc5HQ4Gz23W18LIz4LTPaVhBhK0wegyj8P2NCvhjQ3QASNQeIgHHl0wb/41kZnkw7D27jvF22dGbL6oa+yXZv391ZTfcYGGH3yaaiiBTEoXJOCpGCGOz3DE6DQGknp4OPG7UMCn5t3JeAZdSrr7NcYvE+u7HutV0gho3H955AjfAaQ35OOwsMiGQavWjCwiOMTPNoPYiCKkbfj/4Hh9r5u9UvD3gm019V+bm0MTKZ+sBXQGRV64feuvHQGGOBVjsGhjii5XqURWCDrGnR7vpcP52dJLxGCix2b8LS84QD0/2iyvhKTJU+my/oS8hPeWXd94rHTJqyoemIwcIkS9ZEI/bENA3zmLQ+uH6QaaQCURQ5nF+/ch6jFgl4q9/azpGiIXXOw9jGwiaev8yUrUgRFM5xiktIUAo0syxnQjSO8Zi9ML1B0fppHR0giU+7/u/v2gifMvcN//0sQW5y+0PX9Vhvn73vvrcjQBdJGXciiAIj2/kis0I0j3OZNBh+1hnd3dnJ1IUc3/a+ucC0p8Q/P6P/w4mnODywU+/zyOI6vTdxbdGBFEYYvmXAQRhinQikwGBH+AQizYAExQJ4lQkaOU8tPzJhRDLwj9mvvIjqN13dnjrhfnPYS2dwHw9Z5CvPIB++hbGAj4FgmAs1mzfYJvvIXf2INJV7p7wn8QI2lT/kv97eF9a/9bwFbRDZrf7ySSC0MKBwMUszH92tzLPh9z43Nhom8d3wRUcYTzlbfD/ZQDBNVtf8f7rKIIO/8UvtP/EhHlq78vY77phni0cl76D/UWV6UhjDHK9swJuH7RYF6WLgpB+439k++94BI3//T84/0cHgnb88j/u/TcSvN/66y//EQet8cXb37fPxxfzTRq05NkZDhX6w/eQuzoAqoxRUQCzaeMMimJo715poEFQ1+1AO9sjhHF828CEi/z4h6Lo/VV8Ks9pc31FkxtalOafDmHwADu71sIqHwGiGrOd9RGUhP4Ih/U/2nYI+vIRAwbb54nEXngJTAnUHhzkyWVtDF5HsPULC/QCH5wh/wD+wGFtHqocnksfo0Ir5akiJMjzbY5gJ3xmwmeNdcJz6qYr0dGOR+mcuAdxhHY/UNwPDjCVULI2D7u7YHak+2D/xRZzrA8GJ4lHY0OQ57jFoX7Ik/weGdYGu/oI4Ae+7YBgsyQOgnio6iqhNQ4M3OA3lEchr6sux7iQVzqs0bYdsIjMlDY4yetG73ncPe948vh8m4Ligv/lqGaFgWf/+fZwFg6c7dHxMh9ylqAnqIQDMKWY9Qy1t2N4U2OwHxhG6Gg7wIdrkeO3TIxQuSsXGmQUW3n3q7Aa9sd/8ybf3mZxh+IpNxxIkVkT4EM/nRNjsx1wWHFEIhGDci2Zju6+16Idtc2t6iUVEJZ9ZbePgKLBxXOdFwfQeDq3UsABoAklEzYUQQeVowIa5PEdBBIORei5XGbriwJGWs1XG285GFZ1O025TgwLGv3qDHwRufVaaaUbRY2pgDMFpylLNjA1giIYEX78ePiaSj7v9odzPHXNrs28tmCEqtttSo+hWNKg1caYKFZdXl9aHIPj4fer6hQEZQ8PM6YxBCPB2U/AwHxJNb/87k3/aMvi8D6tYl0bTo8tJsaIaaPB4JFDf4nmdlSIYg5/QNEYBJBnDPMICI7S00UlYUBT0ynjzz5IBC2nxX5+SRpq+ux2nwfrLpkMeocVpaykDosBDYpzOv3zxSmAzTDG+oUUgKP29VC7MOBoaYzhZ5+6Fesul/34dmRq3+12uKq4garVarLEMWrJ9yTjdKAdPm1InpECHIcxOSKlAxytj0qj4oHvyGALPvp0SX3mC9n3bmT8pstjtzbJ4xWn22LOYAMZ42lCH0C7vNKYNqEE+JlxFlPaD3D9/VQoIEDxSmcOXj+v6S9cCfP2kV1YsNh1xvLIZMhoU6vi2EjAuhOU+1CadcZldukAcXpoZJo/Anl6d08/EdQfm4zRh49WLMfeknFnP6lYMrn1pkUhz6f1KTWpbqbbvu1fiKADRp7RZjZC+TE4wuVNAHw/rZvaRwKrn+jtsQcPSs7j4Jpxp1nQLJt9OnXcLnGpAlp1kjnlN2wkFEniiJqvsplMWCdraEQwwwKEfnpXdx8ZXXluVwbPj4q+B0GvcrlUNMbNMZMuElOYFV63OiSZdeoWK+pUz6RKLHEYTR093KFRMX8aRxigd3cPdqLLr2TKzGolG9jV5eK+cNEWMAZssUhBZZB7S7qIRmzRZ+rmwNC0Vjpn1ui7qLxhhnSeRSTS+npoQxSssh1Yy/uTucTdQX3ZHy56Vsy1rVK4otYulI60EZPUaE+WrUEWTyOeVYsNdPrMGEskYZGJlCHW+DAFWzxbWs2mG9ncdfOsEA/nwlfLm/vx+IrGubB0q4u5JRZvasPtn+XLRXyD1DTQN8tViBWsLiJ5mDszSsHV3jXTm8WbVP1xrlxONXLJJzvbDwKZNVVoobmlSYakvmD1JBJbEKr1UofSxOidmbfqlSwakdw/NTNJwa++aS3dNE4Klbt8vBJfL6SfHQZavpV1eULROLbG/PJIZnF7MyaSGFccSYuJ2c+R+s3qaTqRMjDFm+jE117v3Hw4uk7kXhYSj6uHy85X57sfCzvHkkyw+b5x7BXHysmLnRxfZA8uxZUOZt+kJeJw8fqJFPoAj0XB118fv/z86iYVeV5de79/XDKcHy2/b+xv8/OPTp/tPQjOlc53njqaAv7a8VLDGGT28t2yeILfRyANj7MnO/Grr44e/+TmYdpzWT15f3JYVO7Vcu9ax3tTS4/OPoa2HTNrr1rvPfnZqd3b3GnNNkF1ZLzRFI9OGGCxWWNd+OXnW7dv7i4LppOly7cne5W5jXz1zc7u9uDu69an8Zp5ovSm8WYpPjVSudt6cOAcp1jS0WSUR8MPM6dZI1347NXm1ctHJyVRq3j28sF6erBVKD0+qS/jdt83XxRXNJTGh7vX9dhIR+rFo4sj5wAmyybikSkyrn+CyWTQCIHr7ccf3q4Uxzdztx+eL8dwzWLx9ZPtMFr95vh5oyJAW999/PIo3w2iD96+PI71AG44VyjwCBiNCfkemJ9D5733+TQVNNxX10aF5D5Dcm3j9xYbwiH/7x/AdndgtY6AUanG0+Z7+AHLfZ4Tzd9bZEA68ecn/y/yB8OoJJfahzNia6Jth3NH6XaCJItMgXs/PKHi/sGOXsKPQfCDHgBKtaLaViVzoXCinaC5+9vxtmXENgqd0HZxpxxtEPQP0/6c+f8P3sPhadvbAhJ/yDkNzwWGpG2wvdvQ2EwNQTs+bzFD9YPgR3oZ9wSGR//M3w9XYk4orMO8puBxNDBgMDPG0E1DqyzGre3AJZlFGxva7v7BUWKbIxBwf8bbDkDOKBWv9sF6P6lSRNp+DCKdrG1dPvUMtMbltEMEWxzgzE9S2w0S8R3gR3xz/3ABlukge3wUyuABcK+no1Z4yxY3hmeh9SxFDVL44KRGMt3eNsGRyeS2KoM6pi1kyK2tM4NiEaD1ai7qG2v7qayGYLuhgq9sgnH4Yl69CrbPkQmZDMgTe/q6ifcyCGvLo7Gd7V2XbA90nGZCAaUGgPWzvYNUDwD5WqIa7wQg5nTIlV1QPsvF/AnohzQ4QCXf6697/SFej/ldsocdjMu187LcCwg7Z5fnizzYn/XlepEFQMJmlClhuTy7IIfLHtQPw0P9Xe3tGxy+zTuqxSOz9GRauLvzsKksgIH1k6uTZS3A7zTrjTUx1A8ug9rARsC8Ti5mw7KHOjoy0IPB2Ygn4DCAFMsbz/yyVZd9+ezpha5JFpTObq93fSi9tr3V2lcCQtSg0NiECKoyKcS8ITiNxsdH6MS2jOsgQP2xlDx6mZYkjhyly6eX+g2+Knv96Oooh02XD/Y2D2yAGjNJ1C4pgtOZFRLOGAIGJyaGaTAOHLGDAPVPM3zzNit0fx0tPnx8bV6JGhqXd3dXO1TR2tFm/SgIGGELXx5UAYreqJHzWFCeT06N9XVB/UYktfVT1nz9Pjtv+UM+cXV3Z1o6ce1ePX10tjunXT+olfcyGNvvFKuDeoSu0hlkPA6CG2KyRnt77vtPhP3PGa7fheZsf9vIPri4MK98mVy/fXxzdhJwrB1Us7vrdKHXp1AHvWBoQW2Xz8Dycajdfh9U70RSBx5DKoaTS40o8Ke90vFRWLv526XVhw/Pty+bvvJWKrazzVG6gzplMEUclyhdKr4akCA/CssXFE/o6MDhkHXjivypMfPbi8rGgrL08vfra3eVcOr2aaywZQvv7mmNJr9O6Sv3seflbv28uYMyxJweG2SgKIFAJOLx4DSsf/RF7Sd/eNZQPas//e73J/Fjiyjw9utscMsa3TmKmfR+Mc+9MiWckdmNUmtP9+A4a3xoDI8SCVCC4sHTnW+e/vy7L3712ebW6w+/+OXvznWL+4H8x+/jroNU4fRk2anxCebta2IZS2I1KB399CEGgzU4QcKIBDyeSADf2D9mX//y5A9fep8vffXNz/56T1/YDJU/++uE/zgX37ve9WsdWpdyXaWbkKoWZF4mfXCgd2xwvBPyBOgC/dn808L5r4K//lT9Zfo3P3/4x5Yq5zfWPv4p7t1POvbvDrxKbSguXJSbJoRsnUTPow/S6BMDI90YlM8deMgvHBaM18pP3yg+W/zyV5F3u1p/IFjfeRV21HRCd6HgWrB4dNKEycIZ5aim3PNUBpVG75+kwvGHExCPfi3eMClX5VdPpY8jxkvX47ranUieLX60q4wW4XSpaVGI9IKFhsNo4HLHWXEhnTHQ29fLomF4PJ4E+c/lGYV60fq7S8V5Wl0O/H3SlErYN8p/I7fPCwSqxsGCVSjkSesWuYE9PjGbFNGovV1D/VM0DH5/JCIe/SCzSZRh9980dNWAOlb+DwZ/M2/Orv1xLCWSyrTVBntJOC+QN1VTIfbEzGxN0Mnq6xvuY1FR3A/9f67yS1S25N9G3GsWhaP5x/Gl87LR0/yWuL5gWygcZzpfSITzqecCSo3LYgtbUzjpQO9A/xQFLsMEAh4HNp1xidK89a0gv25Smh89J6482HU4316BdVXYuv9Bgz1XiefX3nWjB3NstmirB5EM9vX3ThF+4PEgvFiTaw1/9bhn/5lJbf7PJ1CQPrS7/lMaVGzVyON/PwluHMaFjc/xoGaZ5UtPCAhnYnBwcBoF8P0RCbCuIWtMarj8YwajvrZtxLh5szJ9+jTcujMuhHZP128vSyvN3FJy62rj4Py4lR+orTQPzwtGpMu5vBKYBoCiNsph/YkZzY6VTRXKzOhViZM7e+WxSenePmocnebL1XQhs3K6tntxtF6gLS5t7J1kNQjZVq762rxKJ+qCvN4bLDdE6FhKpw4d3WqSj81q2+ZhfesknouGSrnC8cre+fZ6titdgBkno0I6rMV7nixf4MH6FlX5vbkaA4xkdarAyfGU+YFOp20dVBOHmaV4JJp2Hi41zzdqiyRfZmN/P69C8Pp81Q3Xd7JEPkOA7SviwWSuFwzA9gMnWyPqY4tJsbOzGD6LbUV84aTlILt2slItEZ3Jxt5OWoHgNIWyvc2LF2ZhHYeTrhXi8W7Qmwpr46cr9IVzt0N+tJUKbCY3HW5HyrCTrB1XylmCOVA/2E0pEEyRL9snAUqR6+Zh2sTLDtJRHxnQ0x5b8ChHll3bw4rmRtC3WVhxuawp3W5kdXdxMY03+qqH2wnIL6RLdibkZQYp5HGSZWvYRAA9OYvDth3Ei09Ny6rcitNzmq86bYaUuuWrbGczeZzBWzpsxSEvjxYtEwDtUlsVMH6ctBnxwLqbnNK61GUJOres8i+Y8xLXXnwt6tHlNQ1/vh7KruL03th+M6FBULmvZGFBXuvUEGAdLt9NeDgYoOS1DnmMjc4XFUaJLM4113K5fFGfla1lEy1Pso4ZbK6D9bQeQaXughm+v06NSwV5nKyR8k1hgJTV2qT2cZRXUKhFcz6moRCI5WyatLDRKB15UnXMajLvrieVCCZ1xts8WeOSt3lJMeWbxFBKQmsRq0ZRdkquEvJdowr/gtcm00VmluvLR9FkBTW7TcvrsQUEE9v8Zvj+CEqbAIcCbD6dizAxlBjQKMWqEZQZlyqEQn+/zDTnsEiN3uml+PluIJxBrSV3ZiU6D3mL3cSEE29OC7sNUE4kG2BAP2a9Wq4cRkciWo1E4u0UaebNEYlNzfS59jdscTsWuNiAfqYQvMrq0Y0CBBUoJ1HIT2aLnjav1+kl6gl00K/ViaUR0rxS5FwWmdWTLk+jaY7pcaGbrSOPfwLByewR5SAUIryF8TY/VYq54AFmUjlFhmHQ51WrRGJPh1Aj9G9JzGK23b60Y4orcYmrzRO3dwQhKP0FRW9bMGqZbZ6djwS4kLcawyLLGOj1y9Q8gQc/axdmE3MWNt/iKWybwlJC+nbrJOAeQMi6+LqyzfNsE+2NpOlwwMNGAeoMRpWWYUCPy1R8vh2bsc6nInwbc9YRX9y1h6TE7E3rJOYaRDr16S3TAARZ1rauBayU3dfmPbGQxToCqGGpUcz3YTO6uWxkxj4yFyikGh6fiFy4aB7GHIMIRZfeCrR36LpnaW1+LGTxseBwRlLVVrQf9CR8/7LMs6Ezel14TeKmzyfT4WW7m9udO2puZ2xDCFmVb2UnoaAkTXS1+WGvwQ7lJ+pZLOxmekGXP/X4SmVEZwxJedplo8n8yUJd65zuTh+sbxbMwwhJkd1ITLf/r9d3r4f7nXoYNoJaApVEigY6FytF254J41pt4oWkky6z7LgzNtNkZ3hnrV7SjSAkWbIagh1GsM57PU63mQx02L4hVzGWOwHl0JWTxfRQP81IVDwjXWz2GayzGibFv72yWlYMIx3i+KJjrL2hdr+fh3QZDLpu2L4iFjEskgHZxIvo3GrcpJIjc87Ke+a0Wr2GLZ0gu5ullWXZCNIh8KeMQ+B/FyIEvngWD70JLGqrBo+QRBz9gp2DMWYmZv0yaRdLtWCSSwUMinEpnizyexHihNIio95XSPc8jsUdhRUWmDIKlXMYQuRP8acNY+ggb3TcLBNSxmQLNoWBN0jWpPyRJIcGiMNCtajnX3l0YroXBgImVGwx/KAIvFHmqHoI7eUOjUgFHNLA3ILL7mb3kRRhh9fP6YE8XzpP+9ctVSiMu9r8qGxCMIwieM4QY0ROB1R+/xBHyCT2Tovtbs9ET4fIZTSbmZ0IkcEXCXr/tSYEXX3E9mgMigc50C2ON8AYme8EPcL+AaZ0BNc7zDcaNcPkDoFaoZKPkEDHOH+GS//Rli6Gu4+jo5/aB+s9wOjvHxrEIcSxQXofsxN0Uke5fBYZIwxNMCeGKfDGCKOvG/+jPWWA/jCOFEpnexx7+6h9sE7AD/dTacNE0NE1MD45TABYF71/oAdyuO4u0o/x/1WXolCWtI866d2dMCCsv6+7kwLHg0LvG2jvBxPInV33geLw7Q7/T1BLAwQtAAAACAAAACEA/fz3Nf//////////CAAUAHNsb3QubnB5AQAQABwBAAAAAAAAVAAAAAAAAACb7BfqGxDJyFDGUK2eklqcXKRupaBek2morqOgnpZfVFKUmBefX5SSChJ3S8wpTgWKF2ckFqQC+RqGpmY6mjoKtQpkAy4GAoCRAGDCC5jxAhYCAABQSwMELQAAAAgAAAAhAE2Kcu3//////////w0AFAByYXdfaW5kZXgubnB5AQAQALgBAAAAAAAAzAAAAAAAAACljs1OwmAQRU+xVKgIWEr5s/ARNZ8k3UgCC8PancYNC1amgRJIDJiWuCE8BS/soAl0LbOYZO7MvXP2b+PX94nBN1s9i5JprJ+VHi37OlB6vo43cbj6WMez6KC/hJ9JJHqyCL8imR+fBsOgF6id+nfZXRQd2vjc0qJJnRoeVVwqONxQpkSRawpcYZPnEossJhdkMIBz/UjPyGSKask2Jze23BbEUxRvWTIcyXIl05PsOg350pJv/lneB+65S9Ef2Bu/9Gn2E3nuSP7H/QNQSwECLQAtAAAACAAAACEAfW+mVTwiAACAJwAACgAAAAAAAAAAAAAAgAEAAAAAdGh1bWJzLm5weVBLAQItAC0AAAAIAAAAIQD9/Pc1VAAAABwBAAAIAAAAAAAAAAAAAACAAXgiAABzbG90Lm5weVBLAQItAC0AAAAIAAAAIQBNinLtzAAAALgBAAANAAAAAAAAAAAAAACAAQYjAAByYXdfaW5kZXgubnB5UEsFBgAAAAADAAMAqQAAABEkAAAAAA=="}, "1.2.826.0.1.3680043.8.498.10018552945042470316597008531304016339": {"jpeg_sha256": "337fb18f5e3c40c97233b327c1f850c884ada1246d7fab0796562d090dec7ab7", "n": 158, "arrays_b64": "UEsDBC0AAAAIAAAAIQAyI7hK//////////8KABQAdGh1bWJzLm5weQEAEAAAKAAAAAAAAMggAAAAAAAAnXr3W1pZ1/baiMaYYnpPJt1kYqoxGo29K4qoCChIE1BBkGoBEQQsSO+I2DXG9BnTqzGTyczzXu/3L337kPlhnved7/nh4/K6Fuecfa+9djnrvtdGfwO9vpGFQAf6jE6RWqjKyDudYdDezMg8nSFWqDQqvpyrUHWKiPuVfJlahO+ru/h9Inx96WZ2buZp4u9y5unh0/8/nzRAGQxKTzIA6U41hY4AkvMKqmqxRVfzKwgLh4pKqCnYkrKr6tOxRSeraYexhe113Ufx5V2miI+vdzbWlXF2ApygVeTRd2A/daUFtWn4+c8FhfV7cPPUqoamIwQ+s1P8E4E/ILl/GiC9lf9EXgCQw9E7OBmAiqaGtc34/sVeq4Z/DiClaSBMP49hl/IUlFsYltzxTJ6X8LOyng9wtMVyv68OUAHrkYd3E1CpvpQlwX6ydfWcrgsAu4VSJvcqbp5zq6I9F+N26mKsagJftDpThfvhPl1WchCiiEei8lxAVdrrheZrgO71FJZpfgY4JeVVqO7g5k33StklGHdU565kEPPBWgviCcvu9i9pOxGiC1vM9mIg12izssdvYO9d2XeNOQDXOZwKHB8kcfOym2sxLkM+eo9PTLRqxcxHqEHf6p/SJqcwVNcLPCy0r1SXnztER0mVvTdvNLQBVJXTKuRiBDvZd260dCFAWdWaom4SwD57tL6HjJj9WWLf0M4dNH1JmUuBjhUPFhQMdqFt1P7sm1QR9l/Kqtdi/D5G9k1mP8bnlQwXjeIFPWFfrTVsJ7H6q+r9lr17q0aqy+160k8F+sJ8mQalUfpvX2vtQqilWtCgVyE40ZJ9k20g4wnO6Su34YU9o3xKsexO5kjpNK/r1BGqoZ5i8CRdytOVVEgG0YGagdwbQg1CLKa6ftSSAlcpd25KLTsgqTa/XTK1H+Dn4lmt/UgKR8xu8ngyTrSM0GjaUOqVQk15jVROOtSoLc6S28iojaVtcFh3wLWGu9mmid1AphZ2eB14w90oDq64TqZ0iIStk75rJzgDdHqvZeeVQmUVVdVFPtyqrssdsKchJlVV65pKh+vN93K8jr1AphWxIlPHAO5UmJbdp1OEkm6WcfzuSYWawZDJ92dW9tXSB8U7Dwp1reXmqb2I0dRT4vcegGu8woLo1EFIZlawglOnAFXUaWLeC0kdMjFnQFd90NzZwVaITmbI2pqYg+LD6Uq51Or2HEftbF5lePYcXO4tKwk6j0GysLxjPnAGUA21Ox7OJPUoO/nDA/Xp/e3tPIPkzOmxpma1SXB8h44rHppynEKNXEGLf+4cXOwupU6HTgCZV92x4j+N8bSuhfDPSGrqlIwamrYr+EzFpOjCEVcrY8TEOpusYPO1eus5VNvFL3VHzsD57krKbPg4kIUN/EU/7r+Kzp4PZSL+eJPCYmcikYzSHxy4TNY0NtjWhGcRm061zJkuQkEvt2Nm7gwcE1KoD+InIInXJFubxnhqS9vD+FVgOGtlPhcLsWTUwaXhy0jQUuf6teMsorU2jD8yX4RcJV+6iNsf5tc0rcYwvqNBcT/6E8Y38h7EMqEhUCz3TLUBRUfR+kYuo0YGZTLQfhZVsWq8a6MXIKuPM7gaOgP7BLXUxdBJIPEpipUwTiCNDZ33pzOhcKV1wOemw63JUkV88gLcYNT4/R0/QRanamBNfQ7OyRjKh9FTkMqroazEjgPiVRrXgycBGsqM67FLkLM4Mhz0tcDP7hLNnOsinGdV24KCU5AhqJCuKM/A0a4W2QPcbzK7krYaPQKovdi+FjoOUJs9/SByCW7NuQweFw3OB5i6uOsCHO2oHfR2HoejkirusuIU7Olq6V5xHYckdgVzOXwIEKvKuxjEibCGP7sSugCXYx6Dx10Hx8I96gXHGUgXUDQ+8RFI765suS87Cqlias/a+BE87jLhmn8/ICZjdiF4EOMdM/fDZ+BQxD/o8pVCmj+gWXDgfjqpumB3OiT11NMXRbuA1NnUtTaWDqiDorw/uRMQ3bgy59wNULk8vxA+Cju9bpXTUwBkS0w26z4EJB5VE+lNBZKome7uTAESt4W7asU80dbaf39qO6AW29rSBE7wFdNzURxHitunnbLdBpJuVh2z7wXEbtbFepIAMZhMCwfnG3qz+IER5w06z7o6iYmm2fFwaRznn6ro8vQkjkvu1I1M4IRNH+u3a1IA1VIMtmaMq6GIBES+LOvpX+rE+bJSObUwjG2tfi1uwLbYtjI3gduL/EajAe8nmsHcrybjeemx86kYV9M1KSnCttBg8rViWzrgWVFiXN3g41mcz6DUuj43iuOkWwcM5qOAKnUjwzqcV0sGxxWNuH3+gIt3G9vbGrWhHNs8uX5VAkQcDxbE2BYZHszgfAo1Q/1jozif5RusVuI6p98qJ+K+0d8vuoTtFV1bB8Fbt3X8hQ4iDpV7Fed1KBh5MNuD298Y1BvHt2MiGnF6B/H1T0a3pxQ/PzbsH8Zu4bB+TLQX21MDvYZmbG+qrXNEPHcHlhdo2J7TD48T83pk1BUg+j9gdrmL8f10k7VvN8F3QypWMuFnsDcxjstmo/settmWpdkKbKt+n3yehPmt+tvyIh4+qfjLrwsE/+d++jhH8P++J19fkwi+0379RvA4yt36eJ7g713vZXh+kPl732P8OqQsv2v+jLfFgbfLoq1tmO8i4bFveJlRvSP8fwjdQNbMrFwg8BI9l+gfrtrKynB4L4YaP3FwOI2q8KcivOwd2rFfME+fH/KyLdnYj2fC20bMxy1rdzkL26S+Arkk4edumRBPr+U6ZUOD01kp0/uxFd8NP3ZZGwAVBkKaWgpAmme9rxXPN6qZHad2Y9wORfWonMBLs+uwrVQ3Nz524NfKG4z65YDUK298DTyA8uVX0Uo8vwcWn/ar8Xoj7q/rqgFiXeStPj0xH7J8jh6ByDvS5ppDqO/xTLhHA0mjq18XK5mAmtc+veLV4GVbfjAqlOLmqo9vRh3EflB3rk0S41DkSW0IvCtTmk5jUvLYw6hSxYb9E4HPH5R1QDLEX3wfwetzeSNiZY5itaX7/tZsx3Fc8cuezuOFOd5WNhZBKBCVcES923cOrX906ZlwzOr//a2lEpJtoadb7nq8vV6Fx1kTAHtGfn9vmiL4/7nuoQ/jM2h8rycZ+XxL8l7BngPK+1+jlhZ0ftL7/aW3BFKngmufvd1Y3r2J2drGEBwd/fZxZBLjKe8Glj14Ya/VssO2PUkTgXfhvvozZ+Wxz545GroSs3x9MKdA6Z7IzIf+cYTKP6wONQWSINP66Q17dBsgwTe32ITz101daZvyQtJE+ENYWXHlknr6i/upgHxtyfJtyT+D9vgD0Y+SSazL3j/pp3q3wy375kadYTeQpN+maAMn8P6Pl7G7ryY7Zjcjo9V3MoenN8MfA9uvrtp+Wxr/Be2fdobeMMdSEH3z/nCdZR9kebcel/QfBFL/14lS1Rm8wO/q6cLsJMf0l/C6tuzy2OpW7N3Hg1fmnd9nLB9S9y1ag9OVxr2o/dO6o1J7HLLiXx8Uq48BaWTLWCG7CED93EFh5yNX9NvMo2/sI56NrQfvt66c7Xz6h+7Z69M7YsYpJX/gIlJuxMfK+ZfgxssvsdKei0DybunLhFcxD21qKuhlyO39FrC9Eu0O/br5dPVr1inK+/+yr3+6lOodtfnE+qug/WXRUtt2A65ubM3miy8BKfRVV9h5A0Cy2V/WUg720Y8RkVuWHH76YsP3Oe9Y/+a/Zu2/3yBHA+7nYkM29N5/qK9uy4IrL77EsrmXgRT+qsrjYnzPZ00BrQoMgvgyXaxF1sBCfGCpgjT55bPR+S0XWZ/5XF3xu8CPrpjKuHfg1JvfFgrbrgIp+kVR1XETYOCRqKCxGgZbJiO0Nh0yjkQf9AcqkPPlW0n0zzyk2Vg20hfygeEPjdeJs+HQm98XatrxuEOfe+rZGG9c4uQ3VoK2RhmqqdeBsm/6nXy8HBmePGmy/XkXcV897S9+kwcUx0S0WHobdr/8Pt/SkQko+FpA5VzH+GnK3aZK6C4rnb9XpgEmb+wlVVsKjU/XxarPt6Fo/ZPuzuptuOHQu+8orwL56Wd/QyfmmWCglSbCwn6gK7uAUQRtjaXxu+VKKBPLP1QriyDnlzWt4sN1uLX+wZO9fB1O+IdHstV43tffu8tFZwH5NZjRMgD684sKcSJrYBYFcyvluA7o+61MmQfnX64Y+l5fglMb7303ly/C7ohJnWXAuKU39rxOrDt8TDqt5yyAprm6iJsL+aIcf0GlBM7093wt678Ne14v2tSvT8POl28dWQ9OQXLQ0JNlwbpl7rE+W4V1r7eTWduHE65SQiuS3IRjI7fNdaUMSPEJ3pSazgNpbnm899VOIAW+Dta8wjxuGWbemsT87Ysos22pANbJ+rxRXKj1tlSV9p6B3RN3rE3lDbg993mZ8Rggz/xgywvM16PfFaLX+H0f1NfecmKedLrYt82YCIZNFTkE30gZxWU6rEe9OQ5+cznGsbcabXsAGcLq2icYp3gvHn6O7QCvMMuN843FWJczgu3AcMndcexHwim858Z6wZ23NKi7Asju+K6zYyJSsWd7LTg/yqS6pQnCdquKvdjKNyQ5JuLaUXyPeC5QFRa5cJyO4vm4Dc+vYenLPEFY8qGXbg1+LpVPLvdgy375RTaCrfC1NZvAiWNF+WZsO4YLi5y4vUFmGXiNC9met7H4A3zNf/nrupLAvfklQvBjw9YHTzu2jPfK7HFsuU+L7owSzz1383A+ht4FXcsLTNCtX6zyNYxv/GwxEzxT+8228DOhE353+O9iS/naUzqe8FOWRfTPnL17x47bV7xS1T7D+TTr6xA3gu9nfh1WMgle/6YPJ3j/++TMToL33z2YEWBb/oFXKyP0zfN7ecR8XH2jqX+G5/PUN3OHi+D5zUkzoV92f5l7TuB2fvwYJvj71NvfX+N6F+58t0gJ/5Xv7tWoCR7eQ7TC9Si3PGF3TCoI2gd07UTiGhCC//RBRdcT9liXONHwUilWf/iTIij74YdM+o/4XdzmRLubLV3bErKgTEjIFiykuQkg6cL2vzr6Z/w5VV+iv9ISGTFdKIfaeZq4zmXqCNkDuw3nfsCT/tlBvmyQ6A/XU70nEpbSeYWwtXRdIo4bgZoEMOmn5H+KA1XxCTbA+nppJIOwoybVTcK28vQnCUvp6U3Et4d95J/iSKqekBYSwxR8WcgmbOeqn7BJgn49MbEkWqVtFwG7ar/7I459P+bzr2XZSfVb2/HXNKHfRWzXbbTZKOFvu2zCQsis5NbWsTNE80aVJAHcR/mx3qQfcRxkmN0D+Ns+4aBNgO2u2l4VBdu9QoeROCbZzeToiXjIDMpUGuHnTte1BDD5UGJYP3WNx7xYdh8X8k0juIMDtXWdLNzgWPuwhojrSKvQgGUQ7FXS3EQcSZXcbgKITnUfIvxclnPjUaxzz/PbOh14ho/XN3JE2M8ZukjajRucpQq7CD9na1u0ZYQfqmCcWGdScW8NMZ4sGU+NyzC4KuR2OPGDM00tXA3eSOfaWT0DOK7L7E4eMfDrVS1dxLFLZmfHZC62+1nUMWJj3ZNrRUGsO7O6hEI71rU/tzBEGlwOXOO0CYZwesrVyqXqZEAlFLpgCKfBCiV3QILxWcpWdxZObxXa4d4QDVCBoltqy8P+mrg92puAytvUEsNlLKs7B+SGg7hsamgXTODyTaATK7w4HTONXcY+Ei4/dOY+nwiXL0qZ3EIFVNfE68b1A2pptvB0OK56RrcEywCSgMoWjF0D8uCARBTNhG19erUqeBjrXY25z9OPEFuqkJmFgJoahGJJOy7Xakfbu3FcTY1snqoSUqQUdqepFvZaTV1cDw2ODhl0El8VkLrkRplnjIR4HbKuYS1CrZUsNr8H+ysxt/OFQGppaOaI22CXpo7F1/Hg5Litq3NUChl6k1bplQFZLOUxAr4U1Mmi0ybGSaixnMnkmxDi3DMIO3UoiVFXzxHL4KCujsHRqCAjaJYInGZ0yzymUwVGUHI3hz7u9+0jSSjdrKAvmdRQ3Mbi2BGZeU8pE5tRKqOukicaRCdkVc0M3QS6ZjcLGN4xlDtq7+l2+0jJ3cKpoM/7E1nMdK35g7vJTfckw+0+ckprbl+f2E7azamk8IVWdF5YSmnR+kjZw9Z6tsdBKh71MWROf3Jy72DE5/ZfIqun4xOm8OHk9soxizSclsorHNT3eJP2s8srxdIw6SqL1tCkn0LlCgd71OsmlVt8FWZPcBu5zxJyDvluJykH3ZYJ/ylyN3VsSOVPT+2uUamlweRDvLKKbmMo6VaTQMEzTiGGeMIT8DhJzCF/34TPl0yW+jxT486qJIXNqV9yXyep6Y7hYf/+FEWTeki5lHaAw2cYvS7STSrb1DdsRwy13BGZdCDhwIjD6/aTUY9jxRf2MZDS4jB5g0UkWa9Z7Vy4ROrtMuv7Izt26QYsApmPlEERjSgdTlQhHXA6vHZSs8LgC4X9O5DI4fPHglzEt712hWapJMHE6sy05zpJ6pibG/Olp4wuLsSkdvKhAU8sOukmXTG5p71ed+odtdnnDwfPQrvd4/TYOahxcsnrcVNQk+Op2x+4jti+sFMX2oU0IXdkyI62T0XmQmMO0jFPOBoI+PefN036g6HgNVQ/4bR7bO1QMjHu87paoHxy2ev03QDqVCAw5EtDPF8wOGZCJKs/4hsfI+0MBIKegPfQwdEpjyfoPQ/FeBqcVi7ccjkiblcT3LR7XWOBq/h+wG9zpgDd4w6MqhFS+T0+9ygiOYJ+bzCwM9U66fD6/QfhzqTb5x1rg/OT9qjPVw8nTUaHJXIBMl2RgM1GggqHK2jGvCt04HXTAhrCKL+PRBqY9ATC3hS4ZJr0eCeokD5gCpu8hbBNIbcOhw/DIeeMS4WZ9Ibd6zFjZqW4fG4PF0Dk9AY8owi6xzw+jwXBHqvHO+IuBJJ4IjQSvQSovXvQML0dUgxOi5KB86cj7DFh2XdzPOAK4IRcO+n3ObGuabY6PY5enE+1ruDY9M84gWic43O7MW9JFMNY15C4Qyo1JjJyfyRKx/lvz0Q0GMd58qIvND2FE3Ku3Tc9RU0kcvIPPpAWJ2xytO8Hv5za/098978+6G5Gwh6YCyZani4TJngmqf7Ojwak/+wgdbonYa/Jw4lArmntCYI6IGz5oR8OpvzHOC5+nE/0V1pvJo4ZoPnPZwnCvz1sSfhLrfmhY9D/Iw624/kO4nFDY0IoIPriNMF3iL24lvB3XkFJAEkHkv4pDjRQ4swkrEhkI2Qe4puniOMKNPktcUyCmrqsifjSOg78UxzInCUifk5AYf0QnbCGHiMh30irn94WJa77HiTm48riDz1F2kH6exzbQyMdhLzdtrFiII41yOtyLZe4Xn3yjFjepPjq28tE62btYAK4q3D73+O4sPhJ48ffTnx4u27Eds9rp48QgEeX789z8PPd7/7rBcHfpN6mRUKfoVt/8T85PeGuKPZpLIYnOvvhfHyCOFb49f4bws+tpeiCHNvLL18tEbJ1r6V//izhh9ejJ4A4IxM0DnS31ebFyqgxFLf6MP+XP/rwfQY3aLjvjk9hfPXCy9UhbO9M/fGQ+H1qv9X9jlgXksLOJOKQ6vu1g7cBuj2fFt2H8HKu//HbS7xjxA+c4VW8YvW/bizZcEPK+63HxLFhpf3he2JeD412JcZj7n0zImnBZYBz64UF71jxr39svvsJlwMPw4vv9wG0vXoSm8bjU/y29XoZx6UzfVg0Yz/tDPNGDuZ/u3QrQMfvocm3+XkAL6Ds1e9bG/h971+ff/w2G5Dycdy7dAi329p89yEdb4vR7/43qXjZax/FJrC/yZ4vS/VGfD+09U2D6w715p+/PxcAsj2deLhBBzQR9gfDeYBinz99enkHl//mr64/bkHyWN2maxPzv6l3c77BgZAjsvWlbwrQ2Jfvvz/rB+R9qZ171IvLbd9yxMkGcvztp9+ecSDdZfji/iiCk0rKpvEzG5d3ivdOagChhfkPr7pn8Tb87bfPD7Gf8GtdZMmK+w0/jxi1sCvybvPb/SHI6FdvTa9aIJfd8Nl43wEkc+9YXW0sBT2bsw6KHyI09+XNL/djCIXfDEaWQ0Caj74MDzrhmOf9b5txP9yVyr7fj9uB0lj9ccgagqQIj6mtix9Az0Y0zf2bZBT96JpY3EDItTE0/XANUuKhjdDwEmQEfv32fm0aqgTCP6KxFUSrrvulS7OJyGviZ3PV8+fQS8b7Kfe3dBR74fa7vyQlOV/pVh6+RrtjwXm3/jHcDq9/ebscR7Q+9WL/5EfUUEMbqRF/I5OfPXm2QFm5TPrkff3r0p/HSauPMF982548825s9ckmOhryuFymh6jUGp9+vvQK8SYclHb1FmpktWWVtX9LTnrybukx+2E2evj5/b/e/Osiafp+QGX6np4ce//LxrOtpJ9CnqH4yDpqMARVc/4vSBe3t3BEn5FA2VVBa/pMJkVsQ1HfMwrybUz+9+YfucgzbRZavx8ixx7FJpefkU8F3Lo1y2tEHTMJHCPvkNZro3O475FcpapnNT1PQq7ervtvnnQge3zkz61vNWjSquW6X5wnOQ1mtvHltv1+aY+h/x3KdyiaBMYXSGjQa0dYb5C4p5vFbV5JA5vIEXy+KkPjHsPWb++qkEmj0i49voGG6e3N0j92kt2sukbNB3TKIhIL1W9QQa9YPtL6aVsNvbGlvWn2Ehg6eoXusQ7QjztsK/fbQNMvnlj4NRfUXAaT93EXcraz2cOvUFqsVSru+ojOjrSIZMwPB7OYda3ttOlcUEi4DTaDCKTDEZo/1g0ivWbY+SAXekR8Qd+bNHAxRHz7cyDFmnr7RO/QrmmaSMl8d+SUqobeTp3JhE4lu2ZwqA84w37aaEQEtJHgSP+DLGjVdPFdb1PA0irm2hYAItTert6ngBYonT30z7vSbOWNjNqlY9Cmbq/VmHugZsJBG5lhQ5brl/GejZ8hVy3hPHhBwvEJ2VYLfk8Z/E71LMBMPUdIe09C9lIaq/rxNigf49YZHWw4/srdrF2vgbSt31yTm8fgmKtT8jiO8854Z/tcG84ryg6+2QBg5XG4bcsAE5UtjOpVzO+PpRTuSi2QHk3T7F9uALr/x9zqpzRIXjAZNzS47pnhMjYwMRWGOGwfDaDD3M4hzkcUHGYr3Yvz+4qm1vbmOiD3U/bb33HB4sKJfAkXFsvrv7zAiY58n8Uj6rL9rxltaxdwObvR1ubB9X/NGIMxQJxTQNIPPrnQKzuNzRGxPrSA5US9Ry6ec+1F4nHleDwkgvpFZzwY8cJBsyseifQS/GzUMFMJpvnBzxcVYoI/D/KG/csYXxoQC2ad+4BhlBuXML5k2T4Xm/HDbo19KRaRYT5skIlaiQyctiOBvyQTEvi9bJ1jDhfUBSaRyOfYA43dXcPToU7IC4/NR+NhSBMZY4FpCdYp+UwOjWD4XQkegavKHoJv0zu0o3OYaPJGBZ0e+26oEUtMkUgH5E5PxGOzftjG0c76opjZkq5WN1cQQ99/MIG/rh8ghEw6r2cohAv3m0PcTuvYTqgQCQzuIAOyvGOxaMwDKS2qkCPaigno3L3afOKHkhPHE/gcoYH4uWdnu1DtPIZljKaNb7CmQYVKag5EaJDlmZqfnrEDuUEZdMQaCH2VXXQT40mXziX4tIAtIXTEjjaBdBDrvUvilg41xhd16/SuMAVuTprCgbAF1898ly1ShfHHrt8iDh6Srl9J4Mt6JMR5QSpNopBhHrzIqG8TjqXBvQ7D1FSwBq7bh7yRaT2QSvgmW4xQKEczM4jIST9fS2yAPF4PoXu2Vbf3ibBSOU9rwANIg/w6xbgxWAHXbZPT0Vg/kAo7+kxh4mThyMXjif8HOX8lgc9i9BL/lpFS1SLtxTN6obVFJBlMg4KSpmFuoBQyra7FcAwT0j2e0OgmTkiOZBwgdg46eeHH+Q9blkPgKzv4vTj+C5QGlli7He7lVA/yfUWQaYksT4X5gPK6OlWjhJI4ejGdAKIjJxP4a6w+wus2qrSbGP/5+gqmWLUdCnLrHO2+Qrg6bF82Rlpx/yqNrA9PMDqasSsB/Gv9M5sVxPptoysVMnznYlt1S6cqFe5VNU4wPfmQOWSNGfwNGG8Y6uLsJvDn0xLAHT/27wWmmjjfSKVK5XJi/liNbd2abZBfSu9vCeXBdVN4fjRSCqhkzCau30bgL6YmgEk/dOXZDjkhVLezFEMqrDvOt/uCksEUuFesNwtCuXBrwjNv9OQAqvFYBPkE5PCFbT+E3A8heKaVe4h4mxq5ch1elzPyR7MCHcazGV5L5A5kucMRq/saluU+p5hQcOjQhZS/448z2ogXKa2cJe3dBXAu1MUy6VJQKZ+i6gllwR13NGryXISkBr9bcZGAHMz4Mf6/hOjBpkZiIrZX43XH+Iu8Bmr7UAoqr8hni4PZkD0eDdsmLwC5xhMwEsuH9l/88eL+9dlTW0aMZ3sVVSLAgjfzVjmdN4LxhXeorOhtyPVGop7Ji5Bc7Q17iHNVtO/c4b/jd5TdIWZle0WZkofxN3Iq2aLRbagmp6iBFcyCHEfIPz12GVIorlC0mMDvPXv87/jUu5eIfZxGqdCwMT7nbjWjy5qKaFXU2l7vLVSon46vWDLx+jqD04nKZM/Zo3/HJ185QdxNLaszEPjCsmqWYmIHicHhdZhcN1H5yHR8efI2bKe7Q8Eeoqc9Z/4tftKZvQQ+5V7DAAvjS3hVLM3UrqQWSneP234L1fZHZxYcBbCzzeXz6sgJ/MG/49GBxHZKKaTraTsAlQ3WsfWOPSnMRsVAfCo7iWaYnZt318Eujjvo0eKm6H/iUxP7MDmvWVuDF7bMXc/VO/amdnb0DaxO5SYxTLGZ2WArpPM9YY+UwO/9H/gf54nJObTeEvy4KkzlWtz7d2jl6v4H4/eSO2xzC3E/E9IFvpBXQuD3/zv+r31Ivt3YXbCN+LmWzva6DqWbrfr+FXvJNuFEfHY60gbpnd6Qp5vAHzh7EP1vfFIWVZifjMvgeQbbN3Ek3WozD866KlLF4zNz4TAH9kiCIXcCf/jC4X/Ak66Xc26RMf/FuNzYxPG9dodpMO6mbO+2Tc9GwgLYK43E/L14gkgnfj767/gfy3i1kJlJAhIjJpQs2I8e9rqHB2dD9bs0jvjczBwPDg3MLcbUuARLOp+FE9f/BVBLAwQtAAAACAAAACEA3YDwjf//////////CAAUAHNsb3QubnB5AQAQAB4BAAAAAAAAVAAAAAAAAACb7BfqGxDJyFDGUK2eklqcXKRupaBek2morqOgnpZfVFKUmBefX5SSChJ3S8wpTgWKF2ckFqQC+RqGphY6mjoKtQpkAy4GAoCRAGAiAJgJABa8AABQSwMELQAAAAgAAAAhAOEGP27//////////w0AFAByYXdfaW5kZXgubnB5AQAQALwBAAAAAAAAywAAAAAAAACtzU1LAlEYxfH/ZGaOmpnp5FterLgKs1EIIly7M9q0aBVDTihIyky4kT5FX7iTDPMBzLu697nP+Z2fp5fp86vDhq2dhfF7ZB+NHS9G1jf2YxV9RcHn2yqahX/zSbCMQ83jebAO9e4P7x/8gW++zd7HBYcMx2Q54ZQ8LkVKnFGmwgVVatTxaNCkRZtruhh63HDLHZY+h8gfpUIuMQqpcp44l4l0lVqdnfbffEe3liYN/XjaqGmzqkRFybKEkqSCxLzknBqyasqo0VHzL1BLAQItAC0AAAAIAAAAIQAyI7hKyCAAAAAoAAAKAAAAAAAAAAAAAACAAQAAAAB0aHVtYnMubnB5UEsBAi0ALQAAAAgAAAAhAN2A8I1UAAAAHgEAAAgAAAAAAAAAAAAAAIABBCEAAHNsb3QubnB5UEsBAi0ALQAAAAgAAAAhAOEGP27LAAAAvAEAAA0AAAAAAAAAAAAAAIABkiEAAHJhd19pbmRleC5ucHlQSwUGAAAAAAMAAwCpAAAAnCIAAAAA"}, "1.2.826.0.1.3680043.8.498.10064642360075735959136394136507408269": {"jpeg_sha256": "b61edc64958042a11c9e569384c205610de215e9ea457867916acc87bb11164d", "n": 86, "arrays_b64": "UEsDBC0AAAAIAAAAIQD9koYX//////////8KABQAdGh1bWJzLm5weQEAEAAAFgAAAAAAAG8PAAAAAAAA7VdpbxtJet50NSnrsE7LkkiJl0g2zya72XezySbZvG+JokSJuihR1mlZly1Zkg/JHt+2xh6N5/LYM5h7490B5kiyO9gJJlkkO0CyWGwOIAiyQRAgfyNN7YdskHzaz1Mg8DQL9T5vVb/9PFX1JDOUzpX+5EcXf7SLzFY3ZtYRTofsbWGIQ4fMra5vrk+fn1xdn63W+iPTyxtVuX9jfvpCVf5vYSiHjpF/VoduX/dHtQbgmD+4MH2nC6BXPyrlL7eBzpXX4/69HgA82+lQ9RQAuuXsSBwAoBxZmPbKCKsWFsOw/KCYPLegBclyaPlcHgWZJ8mlkWEU+O89KadmUaAoHQ1mzvUBENwfLWRlHtVqLlhWyvHRcqrSK8ebJkbXhuDqnDidL/hOXbztG83m/MqxD4KpQF4AXden0pkBO6ibnbOP5s4C2H2bFNa0AHSMj4TuhGUeaSg5uNJ2JIbyXi6s/irPpzha7HnnIZXxMxFAXGcpKcED7ZLfTBdF0FCKGEOFEQBjU3gwNg/g+uEhwbeg20WZuJ+RXJ9RHollUuhvCu4oyxUbZvZcLpYeAGLJbjSxWWAsWxELkwJ1+aAVYQfl9YzgJDVMLmNkSqBSwg0CCzDMMPeZi/VSbEEX9RpNFB9oShFmrVogQT6A9BtxHnROO416Up4/EkJxIk5OuLAMg8XFNZoP0+5kfM6H4RQppZ2S3oUJLnKAMOq0tP30qmBDMS8NkGGLFaFDANCCm6QTYtrHFP2YLzzjc3kZdyCaZDUk4cG3TLRO5keqAmpAA6TNcsgRdJjCgW/AarN55IJGKRfB5tlCwLaWY8lwmNGJgpsIih4DxbrsqwZCJ6JOZI/G9D7SZaXX2aCcwAtHWLvFQRbkMqIYRoW5HG3J5NxGKWw3cmmrNhBw6yyCuWex19MnILTqGufSOoWJvv71QBzPpd1w0GNz+n0JoCBQN0Yn0FHM6c3YVMyIo98zbezGIlivrerqGFezKo8YaN/wIr3dpmRn64KH0PEeHWBxsyWEJgDscqBBR1w9S1tcAaTdVLVqrB5tq2UM6banzadDDnsXETOeTk+Y1F39llblLGZUETYVQILmfsYZBIC02cK2dNMNqtth1zW3XEfPWPCextaKs8PY39ugipMdFktfQ/c8qurS6hrgSMCgws0d4GwF6Xc5cABEG4o7JLDFEZQNhcGOn6bMRhheCERRCwHDA3GTwYQo4XMxzRldrxJYcmgPamwEyg213u7UAOAkbHYLDg0mSNyIABC/wrp0NgACOwG7lgbAvR6xGlhZhg9H+vrl4S07KY3eDAM4Tpgt1hYA2oN2xKYBRJVzazplmR5lrCq5u/MW3W9qA6D+GCOdssxOXZsSbRZZbqmLjFUlY/cq2Weu6XjEY6ZPgc4rXqO6puZnqb4uGUF1oUdXw8wlLVpD4j6N1uR+emOErqt1+F9lOmqoOaatADI+CF3b/LUDMh+m/2zxd0aodXvr3wb+1gRB3hf/Ov73bRDU++wfPvkegiB48hf/+DMZoZ6Pv/t1Xa1j/vl3IShXEd76zcEgVJ4Sv/7dd3mIW8n903/+RxYClcpv//2fBQgKXf3yZ3/XCUFdl7955zuZDyp+9vL7hIym4/dfvw42At7jrftrdTsp/uOvf76iWBrzfvtf/zILdZ0v3/7+t0Wo7vz58et/SUBQcEz68FMSglo21l6/sy/HJw/X6Debrnmlm0FpvmdfEj8ZDV/o2soKbzxObkGRqZlHV+fLkGHu3q9u7dqhU5PVG+ceZiDIsXKnkL4FQXVbszHxcu8mQc9KbNl9XmCuZ8MT6KrIX72WHDtVzt19dE4sQcLEN1/uzBch3cr2xWh6G1KI228NBy5B0Nn1RR9WtixT1L7IZr1jHPtOOhLxTUenj24GxvoGJz6/l0vsNqXvffXh7sZtyH8wNx/xLUKt0VfeXAstQpBz48YYn6THRc+RyEjRobn81xNcXipdPHz9yLvqHL7y81uxpSPt8OFPny0W361P3hrNl4JVqLf09PmBMAZBzKV7eTLMLK37divCUrz6QfXPr/Br+fGnP333q/gDf+nidweZ4zfw8s7LrVjsUXfow3K6IsxAyLmffPlQmoQg4ehFLhdjbtwYGTh4ek2c/PC9u+9N7WcSt/7m+V+8d887uPjL47s/PnJXNj++tDr5sNf7p5X4SGoNWGc+/eIgNQtB7mdvCnEffunp45npL3Z9uRffXp/76lJAuvqrF8cfX+VSi7948vnVm+Tc0gfbd0IHXb6Plodz0QXIuvzBF7PRDASQD962+zhbdf/51s2XFXvyyV89HHlt3CYdfPtY2p1xBi58fmMzv2Ivbj57eyez3OT9aHMsLcUhZPH5p7PRFATMz484utR95eqjlfM3o30L739y5723RVXkjR8frD2UEP+dj7Yr1wcsQ3tvP5kpFxSRx1vlhJuAdBdef3eOLsjzP34w6L7UsJ9PBQipt2H9/ujq9LqqKfbG3Paz3d523+6r+9Vte3v64ovjuYAbSu7tVWOoDWofuns8J+QhiFzfKDgq0I5IczgPQ5dD9AW3rPv52OJhQoRBOnPuXHGiES76Vl9NJ9sg2+WRpUCyB1JGK+8sezn58y9ld51paFbw0Da3/DWvcRKiluWyP7dq7JA/0+npMeQsBFkrKGlHIahpk8JwEch5s9MTsTMQdLoUusxgsiyIxOV+CNJcE4uj7ZCs33TW2wJB9cuUZJKxbrHIGltluXCbadQhY8vCHJmXeaAUNTNVD/S3X77yGiHb2cufLD6Vt2fj17fMh/LxAYhWXbBZRmfGsVPD1tdeDtR8BFjFRvA/LfN2UHhwQQEyf80YrxQA4H8Zymb8ANQNo8Yh2YAUFO4pOWS7sqxQCzIPXJ+Yd5wcP2oeBpTVS4QrcFmr3HjKegOPlGDoG3+enwXgzLVQvDgv8wwyVLxUm0+EK9pl7JpOTZ/42CmdTKfb5omgMJrXvJb10fgLY/36Yz/rPaoH+ONUjD6Cgb4SpckDeXQhFAlna364Yr9Q81MYG4gqQGjW6w9wqXn6oSRYHRcLXTfzjM55X6vI7qH9jistsHDO0u+8UQe6ChxPrMrHqIEUXZB54M4SkVaBsakwaTeJC4V1gtMaImX3Me7SWUb4joMootUPG2Q/M+t1ax3AOY0J2KwCtFQItz0n5/eNoU4B7Mx6cTviTq9WecysZxKhfQw36BneLNu6tieEtm5KiEG92A/SZU/Adr4N6CftDiSnBPWTAu4ZBA/XgpzbQbtvbgsei5F1jl8X0P5+t4W7HjJoOjmjZoc361WSUbHBuCLq4W6YSVrtxngLrJ72EHiy/sErfNppI9V718SkRk92b1Szrp5es3pw3K9Vn0HVeNXRb+529TZuk3ZSI2jg4agDQ2Iq4JugOGqg7/75mZFJDO+4PjEYitFIx3a+GBb5/vbFQd6UQ82dwbJBbezSdZ1dQJxWFdIHj/N2h5m2wsk8R2M09Wi0mDuQrB0PCgORWVbfcjMZYklc07IV9wezKVNnomCzx87az6hHbZRRZTirGKcQg8lqhYt+uT5m5m4xEZ32GpoPYpJ32qNu2PMHKSmubloOBjAM7W8OD5HuSJeppS/v8MTtzta6Ua/dgyK9YMbnQb1245PxsDcjGBoOcyG2wKoVNyN+JhPU1y2FIyQaQZRkQtSjiKGuM+ITgry+UTlNGl2MvgUej7KsZNY+mwqzAbsBfloMcrxbA+5meW/Sg4Dx0YggZwDm5YiFtxHg9GLQOegwK+CSZFsO9tfD01EmzDlU7y+LBGEyg1fXBY9Vi4Br+wJHyeNj63lv2I6DjvkQNepAAFzk8JhVD+BcnPG5ZZ65sJfDXC1Dy2GGtvSCyIpI471nALnlZ92mPtCzExZIcz+AU6MxCWkHQCr6Baus69AiRzlkHU0EONaCKxruJijKUgeU6xkGVcNAcTBEIyr5mDI+4OvvkY/71kmfpJHR8DiUwE4DgO0IPCefZ8olL0dg4Af/+ME/fvCPH/zjj/YPhTlCtsjjvV/cGr3SC5xvrSYCoRsNB7ejIkaeo+5vllJewaN7MjOYKvvsjUtX8pEQowEwtVjB2mUfgE0S1Swj/+VGckENNE/24sHI5VMrt0J+nCu7D6sjSZbx9D0cyybGBHt9dT8fC3l1AKYro462mg10C2ybHN/36oQ01A2aFm+FJemisnBT4DBi0HI4Uw7wPN53NJCJ53hnfelqMiAFzHL+SspRiwONrO+sjKdGY7aBTqDM3g9J4U2lcDsgX5BHjDdnxpkAjWmOBzKRmB9vGDhMSQHRDmD3TNrRUZu/sjCrkb1MEfbopW6gDB+GI9Keknkl4HE7RkxXpgq+FMvrbuUL/rDX3SheyYdE3gFgc1myn+RX5G9bajwetzmoArDvYiyc2VMwBwncaR7Xb1V4JonT2t2hHBPk6SbXpYGwhMn16836T/IDWLxhq6GdJ8flunAXRFdgV2G7kSXclsX+SsXjIkxU78x42OV1YA1950cSQZtRvr7GRGv7STy9ba6hOS0tyfdVfIl3Mxtw36UU6bKt9+VmCYoyebSpmQjut2FNbZViNOYwyK8tKqK/j0eGu2qoHU4tyxdYU5kX6Spo3ciSbvtCd3iepHwIpyEqcTpicTXVDxb9YYdV9nVJcJ3MH6hJRQ17Jgtp+f7cXRBT3hKoWy7gDtdcp2uedZAIpTGO+8msx3lakSmxMbt8f1YE/MSZk/ytRlDD1koldgqA5ngkHkoAeLLkmXBNtvausT7BSGlVE9FAmne3gdykL+qU16+IiYLqJF7ZcrKRNc09Cst1rE9HxVgcgOSQN4KtNDcvO1irhjW3TfhIP2VvA/GZoOjul/eBREg0Kk/2MMVJ/rrgjLtWzsQAlYsAQI+4CE/mdN2Cx0BafPb6csDN4q5uEJ4K+D1yweuS8RhZf5IZ/v1OqsU7a0BVvEWPXM4xjnNLjaASY3iScSomAyjNmzQgNMPxHlwJGov5iVjzH8afOiPTQcCQjRTl9enGOMYTbgCjMZYnORxUAk6Bs2pBfIoVSKoeNI+WFlNtfxgPlCcPnYFwpBuAsyNelgzXg4EI7yUoEpoIE5yA6kF+kOVdZBM4MzM+mev8X/G/x2bSi8q87aNhjow2gFSCZWmCh8oRhvS6rWA4xrAo0Qq058eLg2rwf1sDyZrlijTnMl5Knn9wmKRZFweGgzTOumxgQqApB94OHGsDiUH9/xNfh7EW+b00JMt+MtQI+JKHoG0sKAkEijksYIoiMZvzDOQ5nxDTRvDfUEsDBC0AAAAIAAAAIQCakun9//////////8IABQAc2xvdC5ucHkBABAA1gAAAAAAAABSAAAAAAAAAJvsF+obEMnIUMZQrZ6SWpxcpG6loF6Taaiuo6Cell9UUpSYF59flJIKEndLzClOBYoXZyQWpAL5GhZmOpo6CrUK5AMuBnTAiAaY0AEzOmDBBABQSwMELQAAAAgAAAAhAI6uE5D//////////w0AFAByYXdfaW5kZXgubnB5AQAQACwBAAAAAAAAigAAAAAAAACb7BfqGxDJyFDGUK2eklqcXKRupaBuk2mkrqOgnpZfVFKUmBefX5SSChJ3S8wpTgWKF2ckFqQC+RoWZjqaOgq1CuQDLgYGRgYmBmYGFgZWBjYGdgYOBk4GLgZuBh4GXgY+Bn4GAQZCKig3QYBBiEEQSPIDebxAUW6gLCdQFTtQNStQFzNQNyPQFABQSwECLQAtAAAACAAAACEA/ZKGF28PAAAAFgAACgAAAAAAAAAAAAAAgAEAAAAAdGh1bWJzLm5weVBLAQItAC0AAAAIAAAAIQCakun9UgAAANYAAAAIAAAAAAAAAAAAAACAAasPAABzbG90Lm5weVBLAQItAC0AAAAIAAAAIQCOrhOQigAAACwBAAANAAAAAAAAAAAAAACAATcQAAByYXdfaW5kZXgubnB5UEsFBgAAAAADAAMAqQAAAAARAAAAAA=="}}')
tmp = Path('/kaggle/tmp') if Path('/kaggle/tmp').is_dir() else Path('/tmp')
procs = []
for shard in range(2):
    args = {'shard': shard, 'n_shards': 2, 'comp': str(comp), 'weights': weights, 'k_infer': K_INFER,
            'img': 320, 'eval_batch_tokens': 160, 'workers': 2, 'cache_dir': str(tmp / f'v14cache{shard}'),
            'out_dir': str(WORK / 'v14_leg'), 'reference_json': str(ref), 'thumb_tolerance': 3,
            'oof_expect': OOF_EXPECT, 'oof_tolerance': 0.01, 'weight_folds': weight_folds,
            'time_limit_s': LEG_TIME_LIMIT_S}
    env = {**os.environ, 'CUDA_VISIBLE_DEVICES': str(shard)}
    logf = open(WORK / f'v14_leg_log_{shard}.txt', 'w')
    procs.append((subprocess.Popen([sys.executable, '/kaggle/working/v14_leg.py', json.dumps(args)],
                                   env=env, stdout=logf, stderr=subprocess.STDOUT), logf))
for shard, (p, logf) in enumerate(procs):
    p.wait()
    logf.close()
    print(f'shard {shard} exit code {p.returncode}', flush=True)
    print(''.join(open(WORK / f'v14_leg_log_{shard}.txt').readlines()[-6:]), flush=True)

In [ ]:
# Submission: mean of the models' per-finding ranks; unprocessed studies get 0.5.
import numpy as np
import pandas as pd

labels = pd.read_csv(comp / 'sample_submission.csv', nrows=0).columns.tolist()[1:]
test_ids = pd.read_csv(comp / 'test.csv', dtype={'StudyInstanceUID': str}).StudyInstanceUID.tolist()
parts = [pd.read_csv(f, dtype={'StudyInstanceUID': str}) for f in sorted((WORK / 'v14_leg').glob('v14_leg_shard*.csv'))]
ours = pd.concat(parts).set_index('StudyInstanceUID') if parts else pd.DataFrame()
fold_ranks = []
for k in range(len(weights)):
    r = ours[[f'{c}__f{k}' for c in labels]].copy()
    r.columns = labels
    fold_ranks.append(r.rank(pct=True))
final = (sum(fold_ranks) / len(fold_ranks)).rank(pct=True).reindex(test_ids)
missing = int(final.isna().any(axis=1).sum())
final = final.fillna(0.5)
sub = final.reset_index().rename(columns={'index': 'StudyInstanceUID'})
sub = sub[['StudyInstanceUID'] + labels]
assert sub.columns.tolist() == pd.read_csv(comp / 'sample_submission.csv', nrows=0).columns.tolist()
assert sub.StudyInstanceUID.tolist() == test_ids and np.isfinite(sub[labels].to_numpy(float)).all()
sub.to_csv(WORK / 'submission.csv', index=False)
sub.to_csv(WORK / 'v14_submission.csv', index=False)
receipts = {}
for shard in range(2):
    f = WORK / 'v14_leg' / f'v14_leg_receipt_{shard}.json'
    receipts[shard] = json.loads(f.read_text()) if f.is_file() else {'status': 'missing'}
receipt = {'version': VERSION, 'studies': len(test_ids), 'missing_filled_0_5': missing, 'decoders': diag, 'weights': weights_meta,
           'elapsed_hours': round((time.time() - T0) / 3600, 2), 'leg': receipts}
(WORK / 'v14_receipt.json').write_text(json.dumps(receipt, indent=2, default=str))
print(json.dumps({k: v for k, v in receipt.items() if k != 'leg'}, indent=2), flush=True)